# Figure 5 — LFP power and the anchoring state

Does the local field potential differ between anchored and non-anchored trials?

**The raw LFP is in `clark2025_lfp`, not `clark2025`.** The latter carries only
theta *phase* (0–2π, wrapping at ~7.7 Hz), from which power cannot be computed.
The LFP directory holds both: `processing/ecephys/LFP/lfp` (1–600 Hz filtered,
in volts) and `Processed/theta`, each 1 kHz × 48 channel groups.

**Channel groups, not channels.** The file ships 48 groups, each an average of 8
contacts (`CH241_CH242_…_CH248`). The per-day `anatomy_M{mouse}_D{day}.csv`
maps CH1–CH384 to brain region, so groups are assigned to ENTm by majority vote;
groups are contiguous blocks almost always within a single layer, so the
assignment is rarely ambiguous. Median 32 MEC groups per session.

**Each group keeps its layer, its dorsoventral position and its mediolateral
position**, the last because the paper's central axis is mediolateral.
`coord_SCs_x` is stored negative, so ML is its magnitude, split at 3400 µm to
match the cell-level analyses.

In [ ]:
"""LFP band power by anchoring state, across every qualifying session.

Raw LFP is in clark2025_lfp; clark2025 itself carries only theta PHASE, from
which power cannot be computed.

DESIGN

Depth is kept, not averaged: theta amplitude varies steeply through the
entorhinal layers, so collapsing the MEC groups blurs the structure worth
seeing. Power is computed per channel group and carries its layer, its
dorsoventral position, and its MEDIOLATERAL position -- the last because the
paper's central axis is mediolateral, and the cell-level analyses already show
medial MEC carrying more of the anchoring signal than lateral. coord_SCs_x is
stored negative, so ML is its magnitude, split at 3400 um to match the cell
analyses.

BOTH absolute and relative power are recorded. Relative power (band / 1-200 Hz
total) is comparable across sessions, since absolute microvolts depend on
electrode impedance and reference -- but it is COMPOSITIONAL: the bands sum to
one, so theta falling mechanically pushes the others up, and four "effects" can
be one. Absolute power breaks that coupling and is reported alongside, z-scored
within session so it can still be pooled.

SPEED MATCHING is the control that matters. Theta power scales steeply with
running speed, and the two anchoring states differ slightly in speed (35.0 vs
34.2 cm/s across the dataset). Without matching, a speed effect would arrive
wearing an anchoring label. Trials are matched by subsampling within speed
deciles so the two states have the same speed distribution, and every statistic
is computed before and after.

The trial is the independent unit throughout.
"""
import os, sys, json, glob, re, time, warnings
import h5py, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
from scipy.signal import welch
import pynapple as nap
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
_g = {}
for _c in json.load(open('/Users/harryclark/Documents/spatial-manifolds/scripts/'
                         'figures/AnchorDynamics2026/lick_raster_by_trial_type.ipynb'))['cells']:
    if _c['cell_type'] != 'code':
        continue
    _s = ''.join(_c['source'])
    if _s.startswith('MOUSE, DAY') or 'INV = pd.DataFrame' in _s:
        continue
    exec(compile(_s, '<nb>', 'exec'), _g)
globals().update({k: v for k, v in _g.items() if not k.startswith('__')})

LFP_ROOT = '/Volumes/INCR-NolanLab/ActiveProjects/Wolf/harry/data/clark2025_lfp'
OUT = '/Users/harryclark/Documents/spatial-manifolds/data/lfp'
os.makedirs(OUT, exist_ok=True)
FS = 1000.0
BANDS = [('delta', 1, 4), ('theta', 6, 10), ('beta', 10, 20),
         ('gamma', 30, 80), ('high', 80, 200)]
MIN_RUN_S = 1.0
ML_BOUND = 3400.0          # um; matches the cell-level medial/lateral split


def mec_groups(mo, dy, names):
    A = pd.read_csv(f'{LFP_ROOT}/M{mo}/D{dy}/anatomy_M{mo}_D{dy}.csv')
    reg = dict(zip(A.channel_id, A.brain_region.astype(str)))
    dep = dict(zip(A.channel_id, A.coord_SCs_y))
    mlx = dict(zip(A.channel_id, A.coord_SCs_x.abs()))    # stored negative
    rows = []
    for i, n in enumerate(names):
        chs = n.split('_')
        rs = [reg.get(c, '?') for c in chs]
        ent = sum(r.startswith('ENTm') for r in rs)
        if ent <= len(chs) / 2:
            continue
        lay = pd.Series([r for r in rs if r.startswith('ENTm')]).mode().iloc[0]
        ml = float(np.nanmean([mlx.get(c, np.nan) for c in chs]))
        rows.append(dict(g=i, layer=lay,
                         dv=float(np.nanmean([dep.get(c, np.nan) for c in chs])),
                         ml=ml, ml_side='medial' if ml < ML_BOUND else 'lateral'))
    return pd.DataFrame(rows).sort_values('dv')


def speed_matched(tr_speed, tr_state, rng, n_q=10):
    """Trial indices whose speed distribution is equal between the two states."""
    keep = []
    qs = np.quantile(tr_speed, np.linspace(0, 1, n_q + 1))
    qs[0] -= 1; qs[-1] += 1
    for lo, hi in zip(qs[:-1], qs[1:]):
        m = (tr_speed > lo) & (tr_speed <= hi)
        a = np.where(m & tr_state)[0]; b = np.where(m & ~tr_state)[0]
        n = min(len(a), len(b))
        if n:
            keep += list(rng.choice(a, n, replace=False))
            keep += list(rng.choice(b, n, replace=False))
    return np.array(sorted(keep))


def run_session(mo, dy, rng):
    lf = f'{LFP_ROOT}/M{mo}/D{dy}/VR/sub-M{mo}_ses-D{dy}_typ-VR_beh.nwb'
    ana = f'{LFP_ROOT}/M{mo}/D{dy}/anatomy_M{mo}_D{dy}.csv'
    if not (os.path.exists(lf) and os.path.exists(ana)):
        return None
    f = h5py.File(lf, 'r')
    if 'processing/ecephys/LFP/lfp/data' not in f:
        f.close(); return None
    names = [n.decode() for n in f['general/extracellular_ephys/electrodes/channel_name'][:]]
    M = mec_groups(mo, dy, names)
    if len(M) < 4:
        f.close(); return None

    bp, cp = vr_paths(mo, dy)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
    E = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/'
                    'eye_anchoring/eye_anchoring_trials.csv')
    st = E[(E.mouse == mo) & (E.day == dy)]
    if not len(st):
        f.close(); return None
    state = dict(zip(st.trial.astype(int), st.frac_anch > 0.5))

    d = f['processing/ecephys/LFP/lfp/data']
    lfp_t = np.arange(d.shape[0]) / FS
    S = beh['S']
    spd = np.asarray(S.values)[np.searchsorted(np.asarray(S.index), lfp_t)
                               .clip(0, len(S) - 1)]
    run = spd >= 3.0
    cols = M.g.values
    read_cols = np.sort(cols)                       # h5py needs increasing indices
    back = np.array([int(np.where(read_cols == c)[0][0]) for c in cols])
    D = d[:, read_cols][:, back]
    f.close()

    rows = []
    for _, tr in trials.iterrows():
        u = int(tr.number)
        if u not in state:
            continue
        m = (lfp_t >= tr.start) & (lfp_t <= tr.end) & run
        if m.sum() < MIN_RUN_S * FS:
            continue
        fr, P = welch(D[m], fs=FS, nperseg=512, axis=0)
        tot = P[(fr >= 1) & (fr < 200)].sum(axis=0)
        rec = dict(mouse=mo, day=dy, trial=u, anch=bool(state[u]),
                   speed=float(spd[m].mean()), n_samp=int(m.sum()))
        for nm, lo, hi in BANDS:
            ab = P[(fr >= lo) & (fr < hi)].sum(axis=0)
            for j in range(len(cols)):
                rows.append(dict(**rec, band=nm, g=int(cols[j]),
                                 layer=M.layer.values[j], dv=M.dv.values[j],
                                 ml=M.ml.values[j], ml_side=M.ml_side.values[j],
                                 abs_p=float(ab[j]), rel=float(ab[j] / tot[j])))
    R = pd.DataFrame(rows)
    if not len(R):
        return None
    # z-score absolute power within session and channel group, so it can be
    # pooled despite impedance and reference differing between sessions
    R['z_abs'] = R.groupby(['band', 'g']).abs_p.transform(
        lambda v: (v - v.mean()) / v.std() if v.std() > 0 else np.nan)
    tinfo = R.drop_duplicates('trial')[['trial', 'anch', 'speed']].reset_index(drop=True)
    km = speed_matched(tinfo.speed.values, tinfo.anch.values, rng)
    R['speed_matched'] = R.trial.isin(tinfo.trial.values[km])
    return R
print('LFP helpers ready')

## Extraction

~18 s per session, ~20 min for all 58. Writes one row per trial × band ×
channel group to `data/lfp/lfp_power_by_anchoring.csv`.

In [ ]:
ONLY = None      # e.g. 'M26D18' for one session; writes its own file
E = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/'
                'eye_anchoring/eye_anchoring_trials.csv')
sess = sorted(set(zip(E.mouse, E.day)))
print(f'{len(sess)} sessions in the ENTm eye table')
out, t0 = [], time.time()
for k, (mo, dy) in enumerate(sess, 1):
    if ONLY and f'M{mo}D{dy}' != ONLY:
        continue
    rng = np.random.default_rng(abs(hash((mo, dy))) % 2**32)
    try:
        R = run_session(int(mo), int(dy), rng)
    except Exception as e:
        print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True); continue
    if R is None:
        continue
    out.append(R)
    n = R.trial.nunique()
    sides = R.drop_duplicates('g').ml_side.value_counts().to_dict()
    print(f'[{k}/{len(sess)}] M{mo}D{dy}: {n} trials, {R.g.nunique()} MEC groups '
          f'{sides}, {R[R.speed_matched].trial.nunique()} speed-matched, '
          f'{time.time()-t0:.0f}s', flush=True)
A = pd.concat(out, ignore_index=True)
# a single-session run must not clobber the full table: that has already
# happened once in this project and cost a 15-minute rerun
p = (f'{OUT}/lfp_power_by_anchoring.csv' if ONLY is None
     else f'{OUT}/lfp_power_by_anchoring_{ONLY}.csv')
A.to_csv(p, index=False)
print(f'\n{A.trial.nunique()} trials from {A.groupby(["mouse","day"]).ngroups} '
      f'sessions -> {p}')

## Figure

Two things decide how this reads.

**Relative power is compositional.** The five bands sum to one, so theta falling
pushes the others up mechanically — four apparent effects can be one. Absolute
power breaks the coupling, z-scored within session and channel group so it
survives impedance and reference differences between sessions. In a single
session it looked as though theta fell while delta, beta and gamma rose; pooled
and in absolute terms, almost everything falls.

**Speed matching is not optional.** Anchored and non-anchored trials differ by
4.7 cm/s (37.0 vs 41.7), and theta power scales steeply with running speed.
Matching within speed deciles removes that (40.0 vs 40.1) and *halves* the theta
effect. Panel E shows the size of what the control removes; anything quoted
without it is substantially inflated.

In [ ]:
"""Figure 5 — LFP power and the anchoring state.

A  band power, anchored minus non-anchored, before and after speed matching
B  the same in absolute terms (z-scored within session and channel group)
C  theta by entorhinal layer
D  theta by mediolateral position -- a negative result, shown as one
E  what speed matching does

Two design points carry the figure:

RELATIVE POWER IS COMPOSITIONAL. The bands sum to one, so theta falling pushes
the others up mechanically; four apparent effects can be one. Absolute power
breaks that coupling, z-scored within session and channel group so it survives
differences in impedance and reference between sessions.

SPEED MATCHING IS NOT OPTIONAL. Anchored and non-anchored trials differ by
4.7 cm/s (36.96 vs 41.69), and theta power scales steeply with running speed.
Matching within speed deciles removes that (40.05 vs 40.14) and halves the theta
effect. Panel E shows the size of what the control removes.

The trial is the independent unit; every test is paired across sessions.
"""
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
plt.rcParams['font.family'] = 'Arial'

FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
A = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/'
                'lfp_power_by_anchoring.csv')
ANCH, NONANCH = '#9d5391', '#5f9ea0'
BANDS = ['delta', 'theta', 'beta', 'gamma', 'high']
BAND_LBL = {'delta': 'delta\n1–4', 'theta': 'theta\n6–10', 'beta': 'beta\n10–20',
            'gamma': 'gamma\n30–80', 'high': 'high\n80–200'}
print(f'{len(A):,} rows | {A.groupby(["mouse","day","trial"]).ngroups:,} trials | '
      f'{A.groupby(["mouse","day"]).ngroups} sessions')


def per_session(df, col):
    """Anchored minus non-anchored, one value per session."""
    s = df.groupby(['mouse', 'day', 'anch'])[col].mean().unstack().dropna()
    return (s[True] - s[False]).values


def star(p):
    return 'n.s.' if p > .05 else ('*' if p > .01 else ('**' if p > .001 else '***'))


fig = plt.figure(figsize=(12.2, 6.4))
gs = fig.add_gridspec(2, 3, height_ratios=[1, .95], hspace=.52, wspace=.32)
rng = np.random.default_rng(0)


def band_panel(ax, col, sub, title, ylab):
    for i, b in enumerate(BANDS):
        v = per_session(sub[sub.band == b], col)
        ax.scatter(np.full(len(v), i) + rng.uniform(-.16, .16, len(v)), v, s=7,
                   color='0.55', alpha=.55, linewidths=0, zorder=2)
        ax.errorbar(i, v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)), color='k',
                    lw=1.4, capsize=5, marker='_', ms=18, zorder=4)
        p = wilcoxon(v)[1]
        ax.text(i, ax.get_ylim()[1], star(p), ha='center', va='bottom',
                fontsize=7.5, color='0.2')
    ax.axhline(0, color='0.4', ls='--', lw=1, zorder=1)
    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([BAND_LBL[b] for b in BANDS], fontsize=7)
    ax.set_ylabel(ylab, fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title(title, fontsize=9, loc='left', pad=10)


SM = A[A.speed_matched]
band_panel(fig.add_subplot(gs[0, 0]), 'rel', SM,
           'A   relative power, speed-matched\n(anchored − non-anchored)',
           'Δ relative power')
band_panel(fig.add_subplot(gs[0, 1]), 'z_abs', SM,
           'B   absolute power, speed-matched\n(z within session × channel group)',
           'Δ z(absolute power)')

# ---- C: theta by layer -----------------------------------------------------
ax = fig.add_subplot(gs[0, 2])
th = SM[SM.band == 'theta']
lays = [l for l in ('ENTm1', 'ENTm2', 'ENTm3', 'ENTm5', 'ENTm6')
        if th[th.layer == l].groupby(['mouse', 'day']).ngroups >= 10]
for i, l in enumerate(lays):
    v = per_session(th[th.layer == l], 'rel')
    ax.scatter(np.full(len(v), i) + rng.uniform(-.16, .16, len(v)), v, s=7,
               color='0.55', alpha=.55, linewidths=0, zorder=2)
    ax.errorbar(i, v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)), color=ANCH,
                lw=1.4, capsize=5, marker='_', ms=18, zorder=4)
    ax.text(i, ax.get_ylim()[1], f'{star(wilcoxon(v)[1])}\nn={len(v)}',
            ha='center', va='bottom', fontsize=6.5, color='0.2')
ax.axhline(0, color='0.4', ls='--', lw=1, zorder=1)
ax.set_xticks(range(len(lays))); ax.set_xticklabels(lays, fontsize=7)
ax.set_ylabel('Δ relative theta', fontsize=8.5)
ax.tick_params(labelsize=7.5); ax.spines[['top', 'right']].set_visible(False)
ax.set_title('C   theta by layer\n(speed-matched)', fontsize=9, loc='left', pad=10)

# ---- D: theta by mediolateral position, paired within session -------------
ax = fig.add_subplot(gs[1, 0])
w = th.groupby(['mouse', 'day', 'ml_side', 'anch']).rel.mean().unstack('anch')
w['d'] = w[True] - w[False]
w = w['d'].unstack('ml_side').dropna()
for a_, b_ in zip(w.medial.values, w.lateral.values):
    ax.plot([0, 1], [a_, b_], color='0.8', lw=.6, zorder=1)
for x, v, c in ((0, w.medial.values, '#1b6b52'), (1, w.lateral.values, '#7fbfa9')):
    ax.scatter(np.full(len(v), x) + rng.uniform(-.06, .06, len(v)), v, s=12,
               color=c, alpha=.8, zorder=3, linewidths=0)
    ax.errorbar(x, v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)), color='k',
                lw=1.4, capsize=6, marker='_', ms=18, zorder=4)
ax.axhline(0, color='0.4', ls='--', lw=1, zorder=1)
p = wilcoxon(w.medial, w.lateral)[1]
ax.set_xlim(-.4, 1.4); ax.set_xticks([0, 1])
ax.set_xticklabels(['medial\n<3400 µm', 'lateral\n≥3400 µm'], fontsize=7.5)
ax.set_ylabel('Δ relative theta', fontsize=8.5)
ax.tick_params(labelsize=7.5); ax.spines[['top', 'right']].set_visible(False)
ax.set_title(f'D   theta by mediolateral position\npaired within session, '
             f'n={len(w)}: p = {p:.2g}', fontsize=9, loc='left', pad=10)

# ---- E: what speed matching removes ---------------------------------------
ax = fig.add_subplot(gs[1, 1])
for i, (tag, sub) in enumerate((('all trials', A), ('speed-matched', SM))):
    v = per_session(sub[sub.band == 'theta'], 'rel')
    ax.scatter(np.full(len(v), i) + rng.uniform(-.13, .13, len(v)), v, s=9,
               color='0.55', alpha=.55, linewidths=0, zorder=2)
    ax.errorbar(i, v.mean(), yerr=v.std(ddof=1) / np.sqrt(len(v)), color=ANCH,
                lw=1.5, capsize=6, marker='_', ms=20, zorder=4)
ax.axhline(0, color='0.4', ls='--', lw=1, zorder=1)
ax.set_xlim(-.45, 1.45); ax.set_xticks([0, 1])
ax.set_xticklabels(['all trials', 'speed-matched'], fontsize=8)
ax.set_ylabel('Δ relative theta', fontsize=8.5)
ax.tick_params(labelsize=7.5); ax.spines[['top', 'right']].set_visible(False)
t1 = A.groupby(['mouse', 'day', 'trial']).first()
t2 = SM.groupby(['mouse', 'day', 'trial']).first()
ax.set_title('E   the speed control\n'
             f'{t1[t1.anch].speed.mean():.1f} vs {t1[~t1.anch].speed.mean():.1f} cm/s → '
             f'{t2[t2.anch].speed.mean():.1f} vs {t2[~t2.anch].speed.mean():.1f}',
             fontsize=9, loc='left', pad=10)

# ---- F: does the theta effect track the pupil effect? ----------------------
ax = fig.add_subplot(gs[1, 2])
E = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/eye_anchoring/'
                'eye_anchoring_trials.csv')
E['anch'] = E.frac_anch > 0.5
pu = (E.groupby(['mouse', 'day', 'anch']).rad_raw.mean().unstack().dropna())
pu = (pu[True] - pu[False]).rename('d_pupil')
tp = th.groupby(['mouse', 'day', 'anch']).rel.mean().unstack().dropna()
tp = (tp[True] - tp[False]).rename('d_theta')
J = pd.concat([pu, tp], axis=1).dropna()
from scipy.stats import spearmanr
rho, pp = spearmanr(J.d_pupil, J.d_theta)
ax.scatter(J.d_pupil, J.d_theta, s=16, color=ANCH, alpha=.75, linewidths=0)
ax.axhline(0, color='0.6', lw=.8); ax.axvline(0, color='0.6', lw=.8)
ax.set_xlabel('Δ pupil radius (px)', fontsize=8.5)
ax.set_ylabel('Δ relative theta', fontsize=8.5)
ax.tick_params(labelsize=7.5); ax.spines[['top', 'right']].set_visible(False)
ax.set_title(f'F   do the two co-vary across sessions?\n'
             f'ρ = {rho:+.3f}, p = {pp:.2g}, n = {len(J)}', fontsize=9, loc='left', pad=10)

out = f'{FIG}/fig6_lfp_anchoring.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)
plt.show()

print(f'\n{"band":7s} {"rel":>10s} {"p":>10s} {"z_abs":>10s} {"p":>10s}')
for b in BANDS:
    r = per_session(SM[SM.band == b], 'rel'); z = per_session(SM[SM.band == b], 'z_abs')
    print(f'{b:7s} {r.mean():+10.4f} {wilcoxon(r)[1]:10.3g} '
          f'{z.mean():+10.4f} {wilcoxon(z)[1]:10.3g}')
print(f'\npupil-theta coupling: rho={rho:+.3f}, p={pp:.3g}, n={len(J)}')


## Result

Speed-matched, paired across 58 sessions:

| band | Δ relative | p | Δ z(absolute) | p |
|---|---|---|---|---|
| delta 1–4 | −0.0015 | 0.36 | −0.125 | 0.0018 |
| **theta 6–10** | **−0.0097** | **1.0×10⁻⁴** | **−0.311** | **1.2×10⁻⁵** |
| beta 10–20 | +0.0014 | 0.20 | −0.191 | 0.020 |
| gamma 30–80 | +0.0090 | 8.8×10⁻⁵ | −0.030 | 0.86 |
| high 80–200 | −0.0012 | 0.013 | −0.490 | 5.7×10⁻⁹ |

**Theta is lower when the population is anchored**, and in absolute terms so is
almost everything else — the largest single effect is the 80–200 Hz band.

**Layer, not mediolateral position.** ENTm1 shows roughly four times the theta
effect of ENTm2/3/5 (−0.017 against −0.002 to −0.004). Medial and lateral do not
differ (paired within session, n = 27, p = 0.17) — if anything lateral is
marginally larger. This is an independent negative for the mediolateral framing:
the LFP does not use the cell classification at all, and it shows no gradient.

> **⚠ The broadband decrease needs explaining before this is a theta result.**
> High-frequency (80–200 Hz) LFP power largely reflects local spiking, and it
> falls more than theta does. The straightforward reading is lower firing during
> anchored trials — except the classifier *forces* trials with no spikes into the
> non-anchored class, which biases non-anchored toward low firing and should push
> this the other way. Whatever is happening is not simply firing rate, and the
> two candidate tests are in the discussion below.

## Does theta shift in frequency, or weaken?

Band power in a fixed 6–10 Hz window falls if the rhythm weakens **or** if it
moves out of the window. These are different claims about the circuit, and the
data separate them because the file carries theta **phase** alongside the LFP:
instantaneous frequency is the phase derivative, with no fitting and no
windowing.

Three per-trial measures — spectral peak (4–15 Hz), median instantaneous
frequency, and the RMS of the 6–10 Hz filtered signal. That last is an amplitude
that does **not** depend on where in the band the peak sits, which band power
does; that is what makes the two hypotheses separable.

> `np.unwrap` must not be used here. The phase array contains ~0.03% NaN, and
> unwrap propagates NaN forward, so a single gap turns the whole session to NaN —
> which is exactly what happened the first time. The wrapped per-sample
> difference is local and costs only the affected samples.

In [ ]:
"""Does theta shift in FREQUENCY, or weaken in amplitude, during anchoring?

Band power in a fixed 6-10 Hz window falls if the rhythm weakens OR if it moves
out of the window. These are different claims about the circuit, and the data
separate them: the file carries theta PHASE alongside the LFP, so instantaneous
frequency is the phase derivative -- no fitting, no windowing.

Three measures per trial, from MEC channel groups, running samples only:
  peak_hz   frequency of the spectral peak in 4-15 Hz (from the LFP)
  inst_hz   median instantaneous frequency from the unwrapped phase derivative
  amp       theta amplitude, as the RMS of the 6-10 Hz filtered LFP -- an
            amplitude that does NOT depend on where in the band the peak sits,
            unlike band power

If theta merely shifts, peak_hz and inst_hz move while amp holds. If it weakens,
amp falls while the frequencies hold. Both can happen.

Speed matching is carried over from the power analysis: anchored and
non-anchored trials differ by 4.7 cm/s, and theta frequency is even more
speed-dependent than theta power.
"""
import os, time
import h5py, numpy as np, pandas as pd
from scipy.signal import welch, butter, filtfilt
import pynapple as nap

# LFP_ROOT, FS, mec_groups(), speed_matched(), vr_paths() and clip_trials() are
# already defined by the extraction cell above. The standalone script imports
# them from lfp_batch and uses __file__ to find it; neither works in a notebook,
# where __file__ is undefined.
OUT = '/Users/harryclark/Documents/spatial-manifolds/data/lfp'
BP = butter(3, [6 / (FS / 2), 10 / (FS / 2)], btype='band')


def run_session(mo, dy, rng):
    lf = f'{LFP_ROOT}/M{mo}/D{dy}/VR/sub-M{mo}_ses-D{dy}_typ-VR_beh.nwb'
    if not os.path.exists(lf):
        return None
    f = h5py.File(lf, 'r')
    if 'processing/ecephys/LFP/lfp/data' not in f:
        f.close(); return None
    names = [n.decode() for n in f['general/extracellular_ephys/electrodes/channel_name'][:]]
    M = mec_groups(mo, dy, names)
    if len(M) < 4:
        f.close(); return None
    bp_, cp = vr_paths(mo, dy)
    beh = nap.load_file(bp_); clusters = nap.load_file(cp)
    trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
    E = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/'
                    'eye_anchoring/eye_anchoring_trials.csv')
    st = E[(E.mouse == mo) & (E.day == dy)]
    if not len(st):
        f.close(); return None
    state = dict(zip(st.trial.astype(int), st.frac_anch > 0.5))

    cols = M.g.values
    rc = np.sort(cols); back = np.array([int(np.where(rc == c)[0][0]) for c in cols])
    L = f['processing/ecephys/LFP/lfp/data'][:, rc][:, back]
    PH = f['processing/ecephys/Processed/theta/data'][:, rc][:, back]
    f.close()
    n = L.shape[0]
    t = np.arange(n) / FS
    S = beh['S']
    spd = np.asarray(S.values)[np.searchsorted(np.asarray(S.index), t).clip(0, len(S) - 1)]
    run = spd >= 3.0

    # Instantaneous frequency from the phase derivative. np.unwrap is NOT used:
    # it propagates NaN forward, so one gap in the phase array turns everything
    # after it into NaN -- which is exactly what happened first time round. The
    # wrapped per-sample difference is local, so a gap costs only that sample.
    d = np.diff(PH, axis=0)
    d = (d + np.pi) % (2 * np.pi) - np.pi            # to [-pi, pi]
    inst = d * FS / (2 * np.pi)
    inst = np.vstack([inst[:1], inst])
    inst[(inst < 2) | (inst > 20)] = np.nan          # physiologically impossible
    filt = filtfilt(*BP, L, axis=0)

    rows = []
    for _, tr in trials.iterrows():
        u = int(tr.number)
        if u not in state:
            continue
        m = (t >= tr.start) & (t <= tr.end) & run
        if m.sum() < FS:
            continue
        fr, P = welch(L[m], fs=FS, nperseg=512, axis=0)
        sel = (fr >= 4) & (fr < 15)
        peak = fr[sel][np.argmax(P[sel], axis=0)]
        rows.append(dict(mouse=mo, day=dy, trial=u, anch=bool(state[u]),
                         speed=float(spd[m].mean()),
                         peak_hz=float(np.nanmean(peak)),
                         inst_hz=float(np.nanmedian(inst[m])),
                         amp=float(np.sqrt(np.nanmean(filt[m] ** 2)))))
    R = pd.DataFrame(rows)
    if not len(R):
        return None
    R['z_amp'] = (R.amp - R.amp.mean()) / R.amp.std()
    km = speed_matched(R.speed.values, R.anch.values, rng)
    R['speed_matched'] = False
    R.loc[R.index[km], 'speed_matched'] = True
    return R
print('theta frequency helpers ready')

In [ ]:
E = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/'
                'eye_anchoring/eye_anchoring_trials.csv')
sess = sorted(set(zip(E.mouse, E.day)))
out, t0 = [], time.time()
for k, (mo, dy) in enumerate(sess, 1):
    rng = np.random.default_rng(abs(hash((mo, dy))) % 2**32)
    try:
        R = run_session(int(mo), int(dy), rng)
    except Exception as e:
        print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True); continue
    if R is None:
        continue
    out.append(R)
    if k % 10 == 0:
        print(f'[{k}/{len(sess)}] {time.time()-t0:.0f}s', flush=True)
A = pd.concat(out, ignore_index=True)
A.to_csv(f'{OUT}/theta_frequency.csv', index=False)
print(f'\n{len(A)} trials from {A.groupby(["mouse","day"]).ngroups} sessions')

### Result: theta weakens, it does not shift

Speed-matched, paired across 58 sessions:

| measure | anchored | non-anchored | change | p |
|---|---|---|---|---|
| **theta amplitude** | 72.26 | 75.35 | **−4.10%** | 4.0×10⁻⁶ |
| peak frequency | 7.338 Hz | 7.380 Hz | −0.57% | 0.037 |
| instantaneous frequency | 7.846 Hz | 7.885 Hz | −0.49% | 0.018 |

Two independent methods — spectral peak and phase derivative — agree on about
−0.04 Hz. That is far too small to move meaningful power out of a 6–10 Hz
window, and the amplitude effect is roughly seven times larger proportionally
and four orders of magnitude stronger statistically. The band-power result is a
genuine amplitude decrease.

## Example session

M25 D23 rather than M26 D18. M26 D18 has the larger theta effect (−19.6% against
−10.6%) but is the most extreme session in the dataset on the cell side as well —
70% of its cells lock to PC1 where the median is 27% — so it overstates
everything. M25 D23 sits at the median on that measure (0.269) and has the
best-balanced states of any candidate (0.498).

All four panels share one trial axis, so "do they move together?" can be read
directly rather than inferred from a correlation.

In [ ]:
"""Example session: the population anchoring state and theta, on one trial axis.

M25 D23 is used rather than M26 D18. M26 D18 has the larger theta effect
(-19.6% against -10.6%) but it is the most extreme session in the dataset on the
cell side too -- 70% of its cells lock to PC1 where the median is 27% -- so it
overstates everything. M25 D23 sits at the median on that measure (0.269), has
the best-balanced states of any candidate (0.498), and still shows a clear
theta difference.

Panels share one trial axis so the question "do they move together?" can be read
directly rather than inferred from a correlation. Mean running speed is shown on
the same axis because theta amplitude tracks speed closely -- with speed plotted
alongside, a reader can see whether a theta step coincides with a speed step or
is independent of it, rather than having to trust the matching.
"""
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import median_filter
from scipy.stats import mannwhitneyu
import pynapple as nap

# vr_paths, clip_trials, spatial_cell_ids, TL, NBIN and the colours come from the
# extraction cell above.
from spatial_manifolds.anchoring import trial_cluster_labels
plt.rcParams['font.family'] = 'Arial'

FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
MO, DY = 25, 23
SMOOTH = 5            # trials, for the running means only

T = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/theta_frequency.csv')
t = T[(T.mouse == MO) & (T.day == DY)].sort_values('trial').reset_index(drop=True)
print(f'M{MO}D{DY}: {len(t)} trials with LFP')

# per-cell labels, for the raster
bp, cp = vr_paths(MO, DY)
beh = nap.load_file(bp); clusters = nap.load_file(cp)
trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
keep = np.isin(beh['trials'].as_dataframe().number.values.astype(int), orig)
ids = spatial_cell_ids(MO, DY, clusters)
tn, trav = beh['trial_number'], beh['travel']
dt = trav - (float(np.asarray(tn.values)[0]) - 1) * TL
moving = beh['S'].threshold(3.0, method='above').time_support
n_tr = len(beh['trials'].as_dataframe())
tc = nap.compute_1d_tuning_curves(clusters[ids], dt, nb_bins=n_tr * NBIN,
                                  minmax=[0, n_tr * TL], ep=moving)
L = []
for c in ids:
    lab, _, _, _ = trial_cluster_labels(np.asarray(tc[c]).reshape(n_tr, NBIN)[keep])
    if not np.isnan(lab).all():
        L.append(lab)
L = np.array(L)
frac = np.nanmean(L, axis=0)
Lc = np.nan_to_num(L - np.nanmean(L, axis=1, keepdims=True))
U, sv, Vt = np.linalg.svd(Lc, full_matrices=False)
load, pc1 = U[:, 0], Vt[0]
if np.corrcoef(pc1, np.nan_to_num(frac))[0, 1] < 0:
    load, pc1 = -load, -pc1
order = np.argsort(load)[::-1]
print(f'  {L.shape[0]} MEC cells, PC1 {100*sv[0]**2/np.sum(sv**2):.0f}% of label variance')

state = median_filter((frac > 0.5).astype(float), size=9, mode='nearest') > 0.5
bounds = list(np.where(np.diff(state.astype(int)) != 0)[0] + 1)
tr_ax = np.arange(1, len(frac) + 1)
sm = lambda v: pd.Series(v).rolling(SMOOTH, center=True, min_periods=2).mean().values

TA = ListedColormap([NONANCH_COLOR, ANCH_COLOR])
NORM = BoundaryNorm([-0.5, 0.5, 1.5], TA.N)
fig, axes = plt.subplots(5, 1, figsize=(7.6, 8.2), sharex=True,
                         gridspec_kw=dict(height_ratios=[1.5, .7, .8, .8, .8],
                                          hspace=.18))

ax = axes[0]
ax.imshow(L[order], aspect='auto', cmap=TA, norm=NORM, interpolation='nearest',
          extent=[1, len(frac), L.shape[0], 1])
ax.set_ylabel('Cell (PC1 order)', fontsize=8.5)
ax.set_title(f'M{MO} D{DY} — anchoring state and theta on one trial axis',
             fontsize=10, loc='left', pad=6)
for sp in ax.spines.values():
    sp.set_visible(False)

ax = axes[1]
ax.fill_between(tr_ax, 0, pc1, where=pc1 >= 0, color=ANCH_COLOR, lw=0,
                edgecolor='none', interpolate=True)
ax.fill_between(tr_ax, 0, pc1, where=pc1 < 0, color=NONANCH_COLOR, lw=0,
                edgecolor='none', interpolate=True)
ax.axhline(0, color='0.5', lw=.7)
ax.set_ylabel('PC1', fontsize=8.5)
ax.spines[['top', 'right']].set_visible(False)

for ax, col, lab, c in ((axes[2], 'amp', 'Theta amplitude', '#b5651d'),
                        (axes[3], 'inst_hz', 'Theta freq (Hz)', '#3b6ea5'),
                        (axes[4], 'speed', 'Speed (cm/s)', '#4f8f4f')):
    v = np.full(len(frac), np.nan)
    v[t.trial.values - 1] = t[col].values
    ax.plot(tr_ax, v, color='0.75', lw=.6, zorder=2)
    ax.plot(tr_ax, sm(v), color=c, lw=1.8, zorder=3)
    lo, hi = np.nanpercentile(v, [1, 99])          # a couple of spikes otherwise
    pad = (hi - lo) * .35                          # compress the block structure
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_ylabel(lab, fontsize=8.5)
    ax.spines[['top', 'right']].set_visible(False)
    a = t[t.anch][col]; b = t[~t.anch][col]
    ax.text(.995, .06, f'anchored {a.mean():.2f} vs {b.mean():.2f}, '
            f'p = {mannwhitneyu(a, b)[1]:.1g}', transform=ax.transAxes,
            fontsize=7, color='0.3', ha='right')

for ax in axes:
    for bd in bounds:
        ax.axvline(bd + 1, color='0.15', lw=.9, ls='--', zorder=5)
axes[-1].set_xlabel('Trial', fontsize=9)
for ax in axes:
    ax.tick_params(labelsize=7.5)
axes[0].set_xlim(1, len(frac))

out = f'{FIG}/fig6_lfp_example_M{MO}D{DY}.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)
plt.show()
for col in ('amp', 'inst_hz', 'peak_hz'):
    a, b = t[t.anch][col], t[~t.anch][col]
    print(f'  {col:8s} anchored {a.mean():8.3f}  non {b.mean():8.3f}  '
          f'{100*(a.mean()-b.mean())/b.mean():+6.2f}%  p={mannwhitneyu(a,b)[1]:.2g}')


## Does the speed–theta coupling break down?

Theta amplitude and frequency both track running speed, so a state difference in
theta could be nothing more than a state difference in speed. Two separable
questions:

1. **Offset** — at matched speed, is theta smaller when anchored? (main effect of state)
2. **Gain** — is theta's *sensitivity* to speed different between states, i.e. does
   the coupling itself weaken? (speed × state interaction)

Speed is genuinely less variable inside anchored epochs (SD 6.4 vs 8.4 cm/s,
p = 1×10⁻⁸), and restricted range attenuates a correlation on its own. The
**slope** is therefore the primary measure — it is unaffected by range
restriction — with the correlations shown alongside it. Three controls are run
below: trimming both states to their common 5–95% speed support, testing whether
the per-session Δr tracks the per-session ΔSD, and a circular-shift null on the
state vector that preserves run structure and any within-session drift.

The scatter uses **M21 D22**, not the M25 D23 of the trial-axis figure. M21 D22
sits at the median change in coupling (Δr = −0.109), is well sampled in both
states (121/73 trials) and is significant on its own interaction (p = 0.02).
M25 D23 is one of the 14/55 sessions that run the other way, so it cannot carry
this panel.

In [ ]:
"""Does the speed-theta relationship break down between anchoring states?

Theta amplitude and frequency both track running speed, so a state difference in
theta could be nothing more than a state difference in speed. Two separable
questions:

  1. OFFSET  -- at matched speed, is theta smaller when the population is
     anchored? (the intercept / main effect of state)
  2. GAIN    -- is theta's *sensitivity* to speed different between states, i.e.
     does the coupling itself break down? (the speed x state interaction)

Per session: Pearson r of speed against theta within each state, and an OLS
amp ~ speed * anch whose interaction term is the gain test. Sessions are then
combined by Fisher-z on the correlations and by a sign test / Wilcoxon on the
per-session interaction, so no session dominates.

Speed is genuinely less variable inside anchored epochs (SD 6.4 vs 8.4 cm/s,
p = 1e-8), which attenuates a correlation on its own. The slope is therefore the
primary measure -- it is unaffected by range restriction -- and the r panels are
shown alongside it. Controls in st_check.py: the slope difference survives
trimming both states to their common 5-95% speed support (0.12 vs 0.34,
p = 0.008), the per-session dr does not track the per-session dSD
(rho = -0.01, p = 0.92), and a circular-shift null on the state vector, which
preserves run structure and any within-session drift, gives p = 0.005.
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, wilcoxon, ttest_1samp

# ANCH_COLOR / NONANCH_COLOR come from the extraction cell above.
plt.rcParams['font.family'] = 'Arial'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
MINTR = 15                      # trials per state before a within-state r means anything
# The scatter panel uses M21 D22, not the M25 D23 of the trial-axis figure: it
# sits exactly at the median change in coupling (dr = -0.109), is well sampled
# in both states (121/73 trials) and is significant on its own interaction
# (p = 0.02). M25 D23 is one of the 14/55 sessions that run the other way
# (r 0.33 anchored vs 0.15, interaction p = 0.40), so it cannot carry this panel.
MO, DY = 21, 22

T = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/theta_frequency.csv')
T['anch'] = T.anch.astype(bool)


def fit(d, y):
    """OLS y ~ speed * anch. Returns per-state slopes and the interaction test."""
    d = d.dropna(subset=['speed', y])
    s, a, v = d.speed.values, d.anch.values.astype(float), d[y].values
    X = np.column_stack([np.ones_like(s), s, a, s * a])
    beta, *_ = np.linalg.lstsq(X, v, rcond=None)
    resid = v - X @ beta
    dof = len(v) - X.shape[1]
    cov = np.linalg.pinv(X.T @ X) * (resid @ resid) / dof
    se = np.sqrt(np.diag(cov))
    from scipy.stats import t as tdist
    p_int = 2 * tdist.sf(abs(beta[3] / se[3]), dof)          # gain
    p_state = 2 * tdist.sf(abs(beta[2] / se[2]), dof)        # offset at speed 0
    return beta, se, p_int, p_state


rows = []
for (mo, dy), d in T.groupby(['mouse', 'day']):
    a, n = d[d.anch], d[~d.anch]
    if len(a) < MINTR or len(n) < MINTR:
        continue
    r = dict(mouse=mo, day=dy, n_anch=len(a), n_non=len(n))
    for y in ('amp', 'inst_hz'):
        r[f'r_anch_{y}'] = pearsonr(a.speed, a[y])[0]
        r[f'r_non_{y}'] = pearsonr(n.speed, n[y])[0]
        b, se, pi, ps = fit(d, y)
        r[f'slope_non_{y}'], r[f'slope_anch_{y}'] = b[1], b[1] + b[3]
        r[f'inter_{y}'], r[f'p_inter_{y}'] = b[3], pi
        r[f'state_{y}'], r[f'p_state_{y}'] = b[2], ps
    rows.append(r)
R = pd.DataFrame(rows)
print(f'{len(R)} sessions with >= {MINTR} trials in both states '
      f'(of {T.groupby(["mouse","day"]).ngroups})\n')

fz = lambda x: np.arctanh(np.clip(x, -.999, .999))
for y, lab in (('amp', 'theta amplitude'), ('inst_hz', 'theta frequency')):
    ra, rn = R[f'r_anch_{y}'], R[f'r_non_{y}']
    w = wilcoxon(ra, rn)
    print(f'--- speed vs {lab} ---')
    print(f'  within-state r   anchored {np.tanh(fz(ra).mean()):+.3f}   '
          f'non-anchored {np.tanh(fz(rn).mean()):+.3f}   '
          f'Wilcoxon p = {w.pvalue:.3g}  ({(ra > rn).sum()}/{len(R)} higher when anchored)')
    sl_a, sl_n = R[f'slope_anch_{y}'], R[f'slope_non_{y}']
    ti = ttest_1samp(R[f'inter_{y}'], 0)
    print(f'  slope            anchored {sl_a.mean():+.4f}   '
          f'non-anchored {sl_n.mean():+.4f}   per unit speed')
    print(f'  GAIN  (speed x state)  mean {R[f"inter_{y}"].mean():+.4f}, '
          f't p = {ti.pvalue:.3g}, '
          f'{(R[f"p_inter_{y}"] < .05).sum()}/{len(R)} sessions individually p<.05')
    print(f'  OFFSET (state)         mean {R[f"state_{y}"].mean():+.4f}, '
          f't p = {ttest_1samp(R[f"state_{y}"], 0).pvalue:.3g}, '
          f'{(R[f"p_state_{y}"] < .05).sum()}/{len(R)} sessions individually p<.05\n')

R.to_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/speed_theta_coupling.csv',
         index=False)

# ---------------------------------------------------------------- figure
# common speed support: trim both states to their overlapping 5-95% range, so
# the slopes are compared over the same speeds rather than over whatever range
# each state happened to occupy.
trim = []
for (mo, dy), d in T.groupby(['mouse', 'day']):
    a, n = d[d.anch], d[~d.anch]
    if len(a) < MINTR or len(n) < MINTR:
        continue
    lo = max(a.speed.quantile(.05), n.speed.quantile(.05))
    hi = min(a.speed.quantile(.95), n.speed.quantile(.95))
    a, n = a[a.speed.between(lo, hi)], n[n.speed.between(lo, hi)]
    if len(a) < MINTR or len(n) < MINTR:
        continue
    trim.append((np.polyfit(a.speed, a.amp, 1)[0], np.polyfit(n.speed, n.amp, 1)[0]))
trim = np.array(trim)

fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.0))
axes = axes.ravel()

d = T[(T.mouse == MO) & (T.day == DY)]
ax = axes[0]
for m, c, lb in ((d.anch, ANCH_COLOR, 'anchored'), (~d.anch, NONANCH_COLOR, 'non-anchored')):
    s_ = d[m]
    ax.scatter(s_.speed, s_.amp, s=9, color=c, alpha=.55, lw=0, label=lb)
    xs = np.linspace(s_.speed.min(), s_.speed.max(), 50)
    ax.plot(xs, np.polyval(np.polyfit(s_.speed, s_.amp, 1), xs), color=c, lw=2)
    r, _ = pearsonr(s_.speed, s_.amp)
    ax.text(.03, .95 if lb == 'anchored' else .87, f'{lb}: r = {r:.2f}',
            transform=ax.transAxes, fontsize=7.5, color=c, va='top')
_, _, pi, _ = fit(d, 'amp')
ax.text(.03, .79, f'speed x state p = {pi:.2g}', transform=ax.transAxes,
        fontsize=7.5, color='0.3', va='top')
ax.set_xlabel('Trial mean speed (cm/s)', fontsize=9)
ax.set_ylabel('Theta amplitude', fontsize=9)
ax.set_title(f'M{MO} D{DY}', fontsize=9.5, loc='left')
ax.legend(fontsize=7.5, frameon=False, loc='lower right')


def paired(ax, va, vn, ylab, title):
    for i in range(len(va)):
        ax.plot([0, 1], [vn[i], va[i]], color='0.75', lw=.6, zorder=1)
    ax.scatter(np.zeros(len(va)), vn, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
    ax.scatter(np.ones(len(va)), va, s=14, color=ANCH_COLOR, lw=0, zorder=2)
    for x, v, c in ((0, vn, NONANCH_COLOR), (1, va, ANCH_COLOR)):
        ax.plot([x - .18, x + .18], [np.median(v)] * 2, color=c, lw=2.4, zorder=3)
    ax.axhline(0, color='0.5', lw=.7, ls=':')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
    ax.set_xlim(-.4, 1.4)
    ax.set_ylabel(ylab, fontsize=9)
    ax.set_title(f'{title} — p = {wilcoxon(va, vn).pvalue:.2g}', fontsize=9.5, loc='left')


paired(axes[1], trim[:, 0], trim[:, 1], 'Slope (amp per cm/s)',
       f'{len(trim)} sessions, matched speed range')
paired(axes[2], R.r_anch_amp.values, R.r_non_amp.values, 'r (speed, theta amplitude)',
       f'{len(R)} sessions')
paired(axes[3], R.r_anch_inst_hz.values, R.r_non_inst_hz.values,
       'r (speed, theta frequency)', f'{len(R)} sessions')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7.5)
plt.tight_layout()
out = f'{FIG}/fig6_speed_theta_coupling.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)


### Controls

Range restriction, drift and temporal blocking, each tested against the slope difference.

In [ ]:
"""Controls for the speed-theta gain result."""
import numpy as np, pandas as pd
from scipy.stats import pearsonr, wilcoxon, ttest_1samp
rng = np.random.default_rng(0)
T = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/theta_frequency.csv')
T['anch'] = T.anch.astype(bool)
MINTR = 15

def slope(x, y):
    return np.polyfit(x, y, 1)[0]

# ---- 1. speed range by state (does r attenuation explain it?)
rows = []
for (mo, dy), d in T.groupby(['mouse', 'day']):
    a, n = d[d.anch], d[~d.anch]
    if len(a) < MINTR or len(n) < MINTR: continue
    rows.append(dict(sd_a=a.speed.std(), sd_n=n.speed.std(),
                     r_a=pearsonr(a.speed, a.amp)[0], r_n=pearsonr(n.speed, n.amp)[0],
                     sl_a=slope(a.speed, a.amp), sl_n=slope(n.speed, n.amp),
                     ra_hz=pearsonr(a.speed, a.inst_hz)[0], rn_hz=pearsonr(n.speed, n.inst_hz)[0],
                     sla_hz=slope(a.speed, a.inst_hz), sln_hz=slope(n.speed, n.inst_hz)))
R = pd.DataFrame(rows)
print(f'speed SD   anchored {R.sd_a.mean():.2f}  non {R.sd_n.mean():.2f}  '
      f'p = {wilcoxon(R.sd_a, R.sd_n).pvalue:.3g}  ({(R.sd_a > R.sd_n).sum()}/{len(R)} wider when anchored)')
print(f'raw slope  amp  anchored {R.sl_a.mean():+.4f}  non {R.sl_n.mean():+.4f}  '
      f'p = {wilcoxon(R.sl_a, R.sl_n).pvalue:.3g}')
print(f'raw slope  freq anchored {R.sla_hz.mean():+.5f}  non {R.sln_hz.mean():+.5f}  '
      f'p = {wilcoxon(R.sla_hz, R.sln_hz).pvalue:.3g}')
# correlation attenuation predicts dr proportional to d(sd); test residual
import scipy.stats as ss
dr, dsd = (R.r_a - R.r_n).values, (R.sd_a - R.sd_n).values
print(f'  dr vs dSD across sessions: rho = {ss.spearmanr(dr, dsd)[0]:+.3f}, '
      f'p = {ss.spearmanr(dr, dsd)[1]:.3g}')

# ---- 2. circular-shift null on the state vector (preserves run structure + drift)
def gain(d, mask):
    a, n = d[mask], d[~mask]
    if len(a) < MINTR or len(n) < MINTR: return np.nan
    return slope(a.speed, a.amp) - slope(n.speed, n.amp)

obs, nulls = [], []
for (mo, dy), d in T.groupby(['mouse', 'day']):
    d = d.sort_values('trial')
    m = d.anch.values
    if m.sum() < MINTR or (~m).sum() < MINTR: continue
    obs.append(gain(d, m))
    nulls.append([gain(d, np.roll(m, k)) for k in rng.integers(1, len(m), 200)])
obs = np.array(obs); nulls = np.array(nulls)
nm = np.nanmean(nulls, axis=0)
print(f'\ncircular-shift null (200x, state vector rolled within session):')
print(f'  observed mean gain difference {np.nanmean(obs):+.4f}')
print(f'  null mean {np.nanmean(nm):+.4f}  sd {np.nanstd(nm):.4f}  '
      f'p = {(np.sum(nm <= np.nanmean(obs)) + 1) / (len(nm) + 1):.4g}')

# ---- 3. does the gain survive equalising speed range? (quantile-trim to common support)
keep = []
for (mo, dy), d in T.groupby(['mouse', 'day']):
    a, n = d[d.anch], d[~d.anch]
    if len(a) < MINTR or len(n) < MINTR: continue
    lo = max(a.speed.quantile(.05), n.speed.quantile(.05))
    hi = min(a.speed.quantile(.95), n.speed.quantile(.95))
    a2, n2 = a[a.speed.between(lo, hi)], n[n.speed.between(lo, hi)]
    if len(a2) < MINTR or len(n2) < MINTR: continue
    keep.append((slope(a2.speed, a2.amp), slope(n2.speed, n2.amp),
                 pearsonr(a2.speed, a2.amp)[0], pearsonr(n2.speed, n2.amp)[0]))
K = np.array(keep)
print(f'\ncommon speed support ({len(K)} sessions, 5-95% overlap):')
print(f'  slope anchored {K[:,0].mean():+.4f}  non {K[:,1].mean():+.4f}  '
      f'p = {wilcoxon(K[:,0], K[:,1]).pvalue:.3g}')
print(f'  r     anchored {np.tanh(np.arctanh(K[:,2]).mean()):+.3f}  '
      f'non {np.tanh(np.arctanh(K[:,3]).mean()):+.3f}  '
      f'p = {wilcoxon(K[:,2], K[:,3]).pvalue:.3g}')


## Does the coupling change track the state *within* a session?

The paired panels above compare one correlation per state per session, which
cannot distinguish "the coupling sags while the population is anchored" from
"two pools of trials differ on average". A sliding-window correlation along the
trial axis can, so it is worth asking directly.

**It does not resolve within a single session.** A 25-trial window correlation
carries a standard error near 0.2 and the states come in long blocks (median 19
per session), so a session contains only a handful of effectively independent
transitions. Only 2 of 55 sessions beat their own circular-shift null, and the
median per-session ρ between window r and window anchored fraction is −0.148
(34/55 negative; sign test p = 0.11, Wilcoxon p = 0.012) — a consistent
direction, not a per-session result.

**It resolves across sessions.** Pooling windows by state, window r is +0.147
when mostly anchored against +0.265 when mostly non-anchored (p = 5.2 × 10⁻⁴,
51 sessions), and the same holds for window widths of 15 and 40 trials
(p = 1.0 × 10⁻³ and 3.7 × 10⁻³). So the claim belongs at the level of the
population of sessions, and the example traces below are illustration, not
evidence.

M25 D23 is included deliberately: it is the trial-axis example and it runs the
other way (ρ = −0.13, p = 0.81), which is what a readout that reflects each
session honestly should look like.

In [ ]:
"""Per-session sliding-window coupling: does r track the state within sessions?"""
import numpy as np, pandas as pd
from scipy.stats import spearmanr, wilcoxon
rng = np.random.default_rng(0)
T = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/theta_frequency.csv')
T['anch'] = T.anch.astype(bool)
WIN, NSHIFT = 25, 500

def sliding(sp, am, an, win=WIN):
    c=[];r=[];f=[]
    for i in range(len(sp)-win+1):
        s,a,k = sp[i:i+win], am[i:i+win], an[i:i+win]
        m = np.isfinite(s)&np.isfinite(a)
        if m.sum()<win*.6 or np.std(s[m])==0: continue
        c.append(i+win//2); r.append(np.corrcoef(s[m],a[m])[0,1]); f.append(np.nanmean(k))
    return np.asarray(c),np.asarray(r),np.asarray(f)

rows=[]
for (mo,dy),d in T.groupby(['mouse','day']):
    d=d.sort_values('trial'); tr=d.trial.values
    if d.anch.sum()<15 or (~d.anch).sum()<15: continue
    full=np.arange(tr.min(),tr.max()+1); idx=tr-tr.min()
    sp=np.full(len(full),np.nan); am=sp.copy(); an=sp.copy()
    sp[idx],am[idx],an[idx]=d.speed.values,d.amp.values,d.anch.values.astype(float)
    c,r,f=sliding(sp,am,an)
    if len(r)<30 or np.std(f)==0: continue
    rho=spearmanr(r,f)[0]
    null=np.array([spearmanr(r,sliding(sp,am,np.roll(an,k))[2])[0]
                   for k in rng.integers(1,len(an),NSHIFT)])
    null=null[np.isfinite(null)]
    p=(np.sum(np.abs(null)>=abs(rho))+1)/(len(null)+1)
    # how many independent state blocks does this session actually contain?
    st=np.nan_to_num(an)>.5
    nblk=int(np.sum(np.diff(st.astype(int))!=0))+1
    rows.append(dict(mouse=mo,day=dy,rho=rho,p=p,nblk=nblk,nwin=len(r),
                     r_anch=np.mean(r[f>.5]) if (f>.5).any() else np.nan,
                     r_non=np.mean(r[f<.5]) if (f<.5).any() else np.nan))
R=pd.DataFrame(rows)
R.to_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/speed_theta_timecourse.csv',index=False)
neg=(R.rho<0).sum()
from scipy.stats import binomtest
print(f'{len(R)} sessions, window {WIN} trials')
print(f'  per-session rho: median {R.rho.median():+.3f}, {neg}/{len(R)} negative, '
      f'sign test p = {binomtest(neg,len(R)).pvalue:.3g}')
print(f'  vs 0 (Wilcoxon): p = {wilcoxon(R.rho).pvalue:.3g}')
print(f'  individually significant against own shift null: {(R.p<.05).sum()}/{len(R)} '
      f'({(R[(R.p<.05)].rho<0).sum()} of them negative)')
P = R.dropna(subset=['r_anch', 'r_non'])      # keep the pairing
print(f'  window r when mostly anchored {P.r_anch.mean():+.3f} vs mostly non {P.r_non.mean():+.3f}, '
      f'p = {wilcoxon(P.r_anch, P.r_non).pvalue:.3g}  (n = {len(P)})')
print(f'  state blocks per session: median {R.nblk.median():.0f}, range {R.nblk.min()}-{R.nblk.max()}')
print(f'  rho vs n blocks: rho = {spearmanr(R.rho,R.nblk)[0]:+.3f}, p = {spearmanr(R.rho,R.nblk)[1]:.3g}')


In [ ]:
"""Speed-theta coupling across a session, not pooled over it.

The paired panels compare one correlation per state per session, which cannot
show whether the coupling actually sags *while* the population is anchored or
merely differs on average between two pools of trials. Here the correlation is
computed in a sliding window of trials and plotted on the trial axis alongside
the anchored fraction, so the two can be read against each other directly.

Window r is autocorrelated by construction (neighbouring windows share trials),
so the window-r vs window-anchoring correlation cannot be tested with a
parametric p. Significance comes from a circular shift of the state vector
within session: the shift preserves the run structure of the states and the
entire theta/speed time series, and only breaks their alignment.

Both example sessions are shown. M21 D22 carries the scatter panel of
fig6_speed_theta_coupling; M25 D23 is the trial-axis example and runs the other
way, so plotting it is the honest test of whether this readout tracks a session's
direction rather than manufacturing the group effect in every session.

The bottom row is the across-session version of the same readout, and it is what
the claim rests on. A single session does not resolve the effect: a 25-trial
window correlation has a standard error near 0.2, and the states come in long
blocks (median 19 per session), so only 2 of 55 sessions beat their own shift
null. Pooling windows by state across sessions does resolve it, and does so for
window widths of 15, 25 and 40 trials (p = 1e-3, 5e-4, 4e-3).
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
# ANCH_COLOR / NONANCH_COLOR come from the extraction cell above.

plt.rcParams['font.family'] = 'Arial'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
WIN = 25            # trials per window; ~1/8 of a session, enough for a stable r
NSHIFT = 2000
SESSIONS = [(21, 22), (25, 23)]
SUMMARY = ('/Users/harryclark/Documents/spatial-manifolds/data/lfp/'
           'speed_theta_timecourse.csv')      # written by stt_all.py

T = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/lfp/theta_frequency.csv')
T['anch'] = T.anch.astype(bool)
rng = np.random.default_rng(0)


def sliding(sp, am, an, win=WIN):
    """Window-centre trial, r, slope and anchored fraction for each window."""
    n = len(sp)
    c, r, b, f = [], [], [], []
    for i in range(n - win + 1):
        s, a, k = sp[i:i + win], am[i:i + win], an[i:i + win]
        m = np.isfinite(s) & np.isfinite(a)
        if m.sum() < win * .6 or np.std(s[m]) == 0:
            continue
        c.append(i + win // 2)
        r.append(np.corrcoef(s[m], a[m])[0, 1])
        b.append(np.polyfit(s[m], a[m], 1)[0])
        f.append(np.nanmean(k))
    return map(np.asarray, (c, r, b, f))


fig = plt.figure(figsize=(7.4, 7.2))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.15], hspace=.55, wspace=.3)
axes = np.array([[fig.add_subplot(gs[0, :])], [fig.add_subplot(gs[1, :])]])
for row, (MO, DY) in enumerate(SESSIONS):
    d = T[(T.mouse == MO) & (T.day == DY)].sort_values('trial')
    tr = d.trial.values
    full = np.arange(tr.min(), tr.max() + 1)
    sp = np.full(len(full), np.nan); am = np.full(len(full), np.nan)
    an = np.full(len(full), np.nan)
    idx = tr - tr.min()
    sp[idx], am[idx], an[idx] = d.speed.values, d.amp.values, d.anch.values.astype(float)

    c, r, b, f = sliding(sp, am, an)
    rho, _ = spearmanr(r, f)

    # circular shift of the state vector only: theta and speed stay put
    null = np.empty(NSHIFT)
    for j, k in enumerate(rng.integers(1, len(an), NSHIFT)):
        _, _, _, fs = sliding(sp, am, np.roll(an, k))
        null[j] = spearmanr(r, fs)[0]
    p = (np.sum(np.abs(null) >= abs(rho)) + 1) / (NSHIFT + 1)

    ax = axes[row, 0]
    st = np.nan_to_num(an) > .5
    ax.fill_between(full, -1, 1, where=st, color=ANCH_COLOR, alpha=.16,
                    linewidth=0, edgecolor='none', step='mid')
    ax.axhline(0, color='0.5', lw=.7, ls=':')
    ax.plot(full[c], r, color='0.15', lw=1.6, zorder=3)
    ax.set_ylim(-1, 1)
    ax.set_ylabel(f'r (speed, theta amp)\n{WIN}-trial window', fontsize=8.5)
    ax.set_xlim(full[0], full[-1])
    ax.set_title(f'M{MO} D{DY} — window r vs anchored fraction: '
                 f'$\\rho$ = {rho:+.2f}, shift-null p = {p:.3g}',
                 fontsize=9.5, loc='left')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=7.5)
    print(f'M{MO}D{DY}: {len(full)} trials, {len(r)} windows, '
          f'rho = {rho:+.3f}, p = {p:.4g}, '
          f'r anchored {np.mean(np.array(r)[f > .5]):+.3f} '
          f'vs non {np.mean(np.array(r)[f < .5]):+.3f}')

axes[-1, 0].set_xlabel('Trial (window centre)', fontsize=9)
h = plt.Rectangle((0, 0), 1, 1, color=ANCH_COLOR, alpha=.16, lw=0)
axes[0, 0].legend([h], ['anchored'], fontsize=7.5, frameon=False, loc='lower right')

# ---- across sessions: the level at which this readout does resolve
S = pd.read_csv(SUMMARY)
from scipy.stats import wilcoxon

ax = fig.add_subplot(gs[2, 0])
ax.hist(S.rho, bins=np.arange(-1, 1.01, .1), color='0.75', edgecolor='white', lw=.6)
ax.axvline(0, color='0.4', lw=.8, ls=':')
ax.axvline(S.rho.median(), color='#b5651d', lw=2)
ax.set_xlabel(r'per-session $\rho$ (window r, anchored fraction)', fontsize=8.5)
ax.set_ylabel('Sessions', fontsize=9)
ax.set_title(f'median {S.rho.median():+.2f}, {(S.rho < 0).sum()}/{len(S)} negative\n'
             f'{(S.p < .05).sum()}/{len(S)} beat their own shift null',
             fontsize=8.5, loc='left')

P = S.dropna(subset=['r_anch', 'r_non'])
ax = fig.add_subplot(gs[2, 1])
for i in range(len(P)):
    ax.plot([0, 1], [P.r_non.iloc[i], P.r_anch.iloc[i]], color='0.75', lw=.6, zorder=1)
ax.scatter(np.zeros(len(P)), P.r_non, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
ax.scatter(np.ones(len(P)), P.r_anch, s=14, color=ANCH_COLOR, lw=0, zorder=2)
for x, v, col in ((0, P.r_non, NONANCH_COLOR), (1, P.r_anch, ANCH_COLOR)):
    ax.plot([x - .18, x + .18], [v.median()] * 2, color=col, lw=2.4, zorder=3)
ax.axhline(0, color='0.5', lw=.7, ls=':')
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch\nwindows', 'anchored\nwindows'],
                                          fontsize=8)
ax.set_xlim(-.4, 1.4)
ax.set_ylabel('window r (speed, theta amp)', fontsize=9)
ax.set_title(f'{len(P)} sessions — p = {wilcoxon(P.r_anch, P.r_non).pvalue:.2g}',
             fontsize=9, loc='left')
for a in fig.axes:
    a.spines[['top', 'right']].set_visible(False)
    a.tick_params(labelsize=7.5)
out = f'{FIG}/fig6_speed_theta_timecourse.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)


## Mechanism: does the speed signal itself weaken?

The gain change says entorhinal theta is less driven by self-motion when
anchored. The obvious reading is that the velocity input to the network is
down-weighted — when position is pinned by landmarks the network need not
integrate velocity. That predicts flatter speed tuning in the spikes and worse
population speed decoding during anchored epochs.

**It fails both predictions.** Speed-cell tuning gain is unchanged (p = 0.28),
only 49% of 3,719 speed cells are flatter, and speed decodes slightly *better*
when anchored. Three follow-ups pin down why the naive measures cannot be taken
at face value, and what survives:

1. **Position substitutes for speed.** Speed is 4.5× more predictable from
   position when anchored (R² 0.220 vs 0.048, p = 4.6 × 10⁻¹⁰) — the running is
   stereotyped. So both the decoding result and any ΔpR²(speed | position) are
   confounded: a covariate carrying less independent information contributes
   less regardless of the neurons.
2. **Unexpected speed is the clean measure.** With position partialled out of
   both rate and speed, sensitivity to unexpected speed is preserved
   (0.0356 vs 0.0374 Hz per cm/s, −4.7%, p = 0.046; null after rate
   normalisation, p = 0.36). What falls is how much unexpected speed exists
   (SD 13.9 vs 15.8 cm/s), which accounts for the drop in partial correlation.
3. **The dissociation.** The LFP decouples from speed ~13× more than the spikes
   do (−64% vs −5%). Anchoring changes how strongly the population is entrained
   to self-motion, not how individual neurons represent it.

Run order: `speed_tuning.py` (batch, ~4 min) → `speed_tuning_summary.py` →
`residual_speed.py` (~6 min) → `speed_tuning_fig.py`. `poprate.py` and
`dpr2_checks.py` carry the supporting controls; `speed_deltapr2.py` is retained
for provenance but its measure is confounded — see the note in PAPER.md.

In [ ]:
"""Speed tuning by anchoring state: does the velocity-gain account survive?

Reads speed_tuning_cells.csv / speed_tuning_decode.csv (written by
speed_tuning.py) and asks three things:

  1. Do speed-tuned cells have flatter speed tuning when anchored?
  2. Is any such change a change in SPEED SENSITIVITY or just in firing rate?
     slope falls with rate even if tuning shape is untouched, so gain
     (slope / mean rate) is the measure that separates them.
  3. Is speed less decodable from the population when anchored, with bin count
     and speed distribution matched between states?

Sessions are the unit of analysis throughout -- cells within a session share a
behavioural state and are not independent, so a cell-level test would count the
same state change hundreds of times. Cell-level numbers are printed alongside
for reference only.
"""
import sys
import numpy as np, pandas as pd
from scipy.stats import wilcoxon

OUT = '/Users/harryclark/Documents/spatial-manifolds/data/lfp'
C = pd.read_csv(f'{OUT}/speed_tuning_cells.csv')
D = pd.read_csv(f'{OUT}/speed_tuning_decode.csv')

C['speed_cell'] = (C.p_pool < .05) & (C.r_pool > 0)
C['neg_cell'] = (C.p_pool < .05) & (C.r_pool < 0)
print(f'{len(C)} cells, {C.groupby(["mouse","day"]).ngroups} sessions')
print(f'  positively speed-modulated {C.speed_cell.mean()*100:.1f}%, '
      f'negatively {C.neg_cell.mean()*100:.1f}% '
      f'(shift null, p < 0.05)\n')


def paired(df, a, n, lab, by_session=True):
    d = df.dropna(subset=[a, n])
    if by_session:
        d = d.groupby(['mouse', 'day'])[[a, n]].median().dropna()
    if len(d) < 5:
        print(f'  {lab}: too few'); return
    w = wilcoxon(d[a], d[n])
    print(f'  {lab:34s} anchored {d[a].median():+.4f}  non {d[n].median():+.4f}  '
          f'({100*(d[a].median()-d[n].median())/abs(d[n].median()):+6.1f}%)  '
          f'p = {w.pvalue:.3g}  n = {len(d)}')


for tag, name in (('', 'all running bins'), ('cs_', 'common speed support')):
    print(f'--- speed cells only, {name} (session medians)')
    S = C[C.speed_cell]
    paired(S, f'{tag}slope_a', f'{tag}slope_n', 'slope (Hz per cm/s)')
    paired(S, f'{tag}gain_a', f'{tag}gain_n', 'gain (slope / mean rate)')
    paired(S, f'{tag}r_a', f'{tag}r_n', 'r (rate, speed)')
    paired(S, f'{tag}rate_a', f'{tag}rate_n', 'mean rate (Hz)')
    print()

print('--- all cells, all running bins (session medians)')
paired(C, 'slope_a', 'slope_n', 'slope (Hz per cm/s)')
paired(C, 'gain_a', 'gain_n', 'gain (slope / mean rate)')
paired(C, 'rate_a', 'rate_n', 'mean rate (Hz)')
print()

print('--- cell level, for reference only (not independent)')
S = C[C.speed_cell].dropna(subset=['slope_a', 'slope_n'])
print(f'  slope   anchored {S.slope_a.median():+.4f} vs non {S.slope_n.median():+.4f}, '
      f'p = {wilcoxon(S.slope_a, S.slope_n).pvalue:.3g}, '
      f'{(S.slope_a < S.slope_n).mean()*100:.0f}% of {len(S)} cells flatter\n')

print('--- population speed decoding, bins matched for count and speed')
d = D.dropna(subset=['dec_r_a', 'dec_r_n'])
print(f'  r(pred, true)   anchored {d.dec_r_a.median():.3f}  non {d.dec_r_n.median():.3f}  '
      f'p = {wilcoxon(d.dec_r_a, d.dec_r_n).pvalue:.3g}  n = {len(d)}')
print(f'  abs error cm/s  anchored {d.dec_mae_a.median():.2f}  non {d.dec_mae_n.median():.2f}  '
      f'p = {wilcoxon(d.dec_mae_a, d.dec_mae_n).pvalue:.3g}')
print(f'  ({(d.dec_r_a > d.dec_r_n).sum()}/{len(d)} sessions decode BETTER when anchored)')
print(f'  matched speed SD  anchored {d.speed_sd_a.median():.2f}  non {d.speed_sd_n.median():.2f}')


In [ ]:
"""Does MEC track UNEXPECTED speed differently when anchored?

dpR2(speed | position) fell when anchored, but it cannot be read as reduced
speed coding: anchored running is stereotyped, so speed is 4.5x more
predictable from position (R2 0.22 vs 0.05), and a covariate that carries less
independent information must contribute less. The redundancy is behavioural.

This asks the question redundancy cannot answer. Within each state, position is
partialled out of BOTH sides:

    speed residual  = speed - E[speed | position]
    rate residual   = rate  - E[rate  | position]

and the two residuals are related to each other. The speed residual is
orthogonal to position by construction, so a difference between states cannot
come from position standing in for speed. It is the quantity a velocity
integration account cares about: whether the network registers departures from
the expected speed profile.

E[. | position] is fit within state, cross-validated, from cyclic position
(sin, cos), with the SMALL model -- 50 rounds at depth 3 -- because the 200
round depth 5 model drove every single-covariate pR2 negative.

TWO SCALES, because they answer different questions:

  slope per cm/s      physical gain: Hz per cm/s of unexpected speed
  slope per z         gain relative to how much unexpected speed there IS in
                      that state

The second matters because anchored epochs have less residual speed to begin
with. If the network registered unexpected speed identically, the per-cm/s
slope would match across states while the per-z slope would be smaller when
anchored purely because a z unit is smaller. Reporting one without the other
would manufacture a result either way.
"""
import sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import xgboost as xgb
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon

s = open('/private/tmp/claude-501/-Users-harryclark-Documents-spatial-manifolds/'
         '35f3d8d1-3ead-4d48-8089-491c15b000b7/scratchpad/speed_deltapr2.py').read()
exec(s[:s.index(chr(10) + 'if __name__')])

SMALL = {'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.6, 'seed': 2925}
NR, FOLD, CAP_C = 50, 3, 20


def cv_predict(X, y, poisson):
    p = dict(SMALL)
    p['objective'] = 'count:poisson' if poisson else 'reg:squarederror'
    pred = np.empty(len(y), float)
    for tr, te in KFold(FOLD, shuffle=True, random_state=0).split(X):
        m = xgb.train(p, xgb.DMatrix(X[tr], label=y[tr]), NR)
        pred[te] = m.predict(xgb.DMatrix(X[te]))
    return pred


def run_session(mo, dy, state, cells, rng):
    got = binned_full(mo, dy, state)
    if got is None:
        return None
    Y, speed, pos, anch, ids = got
    ia, iN = matched_bins(speed, anch, rng)
    k = min(len(ia), len(iN))
    if k < 400:
        return None
    use = [c for c in cells if c in list(ids)]
    if len(use) > CAP_C:
        use = list(rng.choice(use, CAP_C, replace=False))
    if not use:
        return None

    rows = []
    for idx, lab in ((ia, 'a'), (iN, 'n')):
        ang = 2 * np.pi * pos[idx] / TL
        P_ = np.c_[np.sin(ang), np.cos(ang)]
        sp = speed[idx]
        sres = sp - cv_predict(P_, sp, poisson=False)       # unexpected speed
        sd = sres.std()
        for c in use:
            y = Y[idx, list(ids).index(c)].astype(float)
            if y.sum() < 50:
                continue
            yres = y - cv_predict(P_, y, poisson=True)
            b = np.polyfit(sres, yres, 1)[0] / BIN          # Hz per cm/s
            rows.append(dict(mouse=mo, day=dy, cluster=int(c), state=lab,
                             slope_cms=b, slope_z=b * sd,
                             r_partial=np.corrcoef(sres, yres)[0, 1],
                             rate=float(y.mean() / BIN),
                             gain_cms=b / (y.mean() / BIN) if y.mean() > 0 else np.nan,
                             sres_sd=float(sd), n_bins=k))
    return pd.DataFrame(rows)


if __name__ == '__main__':
    T = pd.read_csv(f'{OUT}/theta_frequency.csv')
    C = pd.read_csv(f'{OUT}/speed_tuning_cells.csv')
    C = C[(C.p_pool < .05) & (C.r_pool > 0)]
    sess = sorted(set(zip(T.mouse, T.day)))
    out, t0 = [], time.time()
    for i, (mo, dy) in enumerate(sess, 1):
        st = T[(T.mouse == mo) & (T.day == dy)]
        cells = C[(C.mouse == mo) & (C.day == dy)].cluster.astype(int).tolist()
        if not cells:
            continue
        rng = np.random.default_rng(abs(hash((mo, dy))) % 2**32)
        try:
            R = run_session(int(mo), int(dy),
                            dict(zip(st.trial.astype(int), st.anch.astype(bool))),
                            cells, rng)
        except Exception as e:
            print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True)
            continue
        if R is None or not len(R):
            continue
        out.append(R)
        print(f'[{i}/{len(sess)}] M{mo}D{dy}: {R.cluster.nunique()} cells, '
              f'{time.time()-t0:.0f}s', flush=True)
    A = pd.concat(out, ignore_index=True)
    A.to_csv(f'{OUT}/residual_speed.csv', index=False)

    W = A.pivot_table(index=['mouse', 'day', 'cluster'], columns='state',
                      values=['slope_cms', 'slope_z', 'r_partial', 'gain_cms',
                              'sres_sd', 'rate']).dropna()
    print(f'\n{len(W)} cells, {A.groupby(["mouse","day"]).ngroups} sessions')
    print(f'  unexpected-speed SD: anchored {W[("sres_sd","a")].median():.2f} vs '
          f'non {W[("sres_sd","n")].median():.2f} cm/s')
    for m, lab in (('slope_cms', 'slope (Hz per cm/s unexpected)'),
                   ('slope_z', 'slope (Hz per SD unexpected)'),
                   ('gain_cms', 'gain (slope / rate, per cm/s)'),
                   ('r_partial', 'partial r (rate, speed | position)')):
        g = W[m].groupby(level=['mouse', 'day']).median().dropna()
        print(f'  {lab:36s} anchored {g["a"].median():+.5f}  non {g["n"].median():+.5f}  '
              f'p = {wilcoxon(g["a"], g["n"]).pvalue:.3g}  '
              f'({(g["a"] < g["n"]).sum()}/{len(g)} lower when anchored)')


In [ ]:
"""Figure: MEC speed tuning and speed decoding by anchoring state.

A  one speed cell's tuning curve in each state (mean +/- SEM per speed bin)
B  session medians of gain (slope / mean rate) for speed cells, paired
C  every speed cell, anchored gain against non-anchored gain
D  population speed decoding, bins matched for count and speed distribution
E  how much UNEXPECTED speed there is in each state (SD of speed after
   position is partialled out) -- the thing that actually changes
F  sensitivity to unexpected speed, Hz per cm/s -- the thing that does not

Gain rather than raw slope carries panels B-C: a cell that simply fires less
when anchored has a smaller slope with its tuning shape untouched, and the
question here is whether speed sensitivity changes, not whether rate does.
"""
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
from spatial_manifolds.anchoring import ANCH_COLOR, NONANCH_COLOR

exec(open('/private/tmp/claude-501/-Users-harryclark-Documents-spatial-manifolds/'
          '35f3d8d1-3ead-4d48-8089-491c15b000b7/scratchpad/speed_tuning.py')
     .read().split('if __name__')[0])

plt.rcParams['font.family'] = 'Arial'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
C = pd.read_csv(f'{OUT}/speed_tuning_cells.csv')
D = pd.read_csv(f'{OUT}/speed_tuning_decode.csv')
C['speed_cell'] = (C.p_pool < .05) & (C.r_pool > 0)
S = C[C.speed_cell].dropna(subset=['gain_a', 'gain_n'])

# Example cell: the one closest to the MEDIAN change in gain across all speed
# cells. Selecting a cell that flattens would misrepresent the result -- the
# population answer is that tuning does not flatten -- so the selection is not
# allowed to condition on the direction of the change.
cand = S[(S.rate > 2) & (S.n_a > 400) & (S.n_n > 400) & (S.r_pool > .25)].copy()
cand['d'] = cand.gain_a - cand.gain_n
ex = cand.iloc[(cand.d - S.eval('gain_a - gain_n').median()).abs().argsort()].iloc[0]
MO, DY, CL = int(ex.mouse), int(ex.day), int(ex.cluster)
print(f'example cell M{MO}D{DY} cluster {CL}: '
      f'gain {ex.gain_a:.4f} anchored vs {ex.gain_n:.4f}, r_pool {ex.r_pool:.2f}')

T = pd.read_csv(f'{OUT}/theta_frequency.csv')
st = T[(T.mouse == MO) & (T.day == DY)]
R, speed, anch, ids = binned(MO, DY, dict(zip(st.trial.astype(int), st.anch.astype(bool))))
rate = R[:, list(ids).index(CL)]

fig, axes = plt.subplots(2, 3, figsize=(9.8, 6.0))
ax = axes[0, 0]
edges = np.arange(np.floor(np.quantile(speed, .02)), np.quantile(speed, .98), 5.)
ctr = edges[:-1] + 2.5
for m, c, lb in ((anch, ANCH_COLOR, 'anchored'), (~anch, NONANCH_COLOR, 'non-anchored')):
    mu = np.full(len(ctr), np.nan); se = np.full(len(ctr), np.nan)
    for i in range(len(ctr)):
        v = rate[m & (speed >= edges[i]) & (speed < edges[i + 1])]
        if len(v) >= 10:
            mu[i], se[i] = v.mean(), v.std() / np.sqrt(len(v))
    ax.fill_between(ctr, mu - se, mu + se, color=c, alpha=.25, lw=0, edgecolor='none')
    ax.plot(ctr, mu, color=c, lw=2, label=lb)
ax.set_xlabel('Running speed (cm/s)', fontsize=9)
ax.set_ylabel('Firing rate (Hz)', fontsize=9)
ax.set_title(f'M{MO} D{DY} cluster {CL}', fontsize=9.5, loc='left')
ax.legend(fontsize=7.5, frameon=False)

ax = axes[0, 1]
G = S.groupby(['mouse', 'day'])[['gain_a', 'gain_n']].median().dropna()
for i in range(len(G)):
    ax.plot([0, 1], [G.gain_n.iloc[i], G.gain_a.iloc[i]], color='0.75', lw=.6, zorder=1)
ax.scatter(np.zeros(len(G)), G.gain_n, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
ax.scatter(np.ones(len(G)), G.gain_a, s=14, color=ANCH_COLOR, lw=0, zorder=2)
for x, v, c in ((0, G.gain_n, NONANCH_COLOR), (1, G.gain_a, ANCH_COLOR)):
    ax.plot([x - .18, x + .18], [v.median()] * 2, color=c, lw=2.4, zorder=3)
ax.axhline(0, color='0.5', lw=.7, ls=':')
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
ax.set_xlim(-.4, 1.4)
ax.set_ylabel('gain (slope / mean rate)', fontsize=9)
ax.set_title(f'{len(G)} sessions — p = {wilcoxon(G.gain_a, G.gain_n).pvalue:.2g}',
             fontsize=9.5, loc='left')

ax = axes[0, 2]
lim = np.nanpercentile(np.r_[S.gain_a, S.gain_n], [1, 99])
ax.scatter(S.gain_n, S.gain_a, s=6, color='0.45', alpha=.35, lw=0)
ax.plot(lim, lim, color='0.2', lw=.9, ls='--')
ax.set_xlim(*lim); ax.set_ylim(*lim)
ax.set_xlabel('gain, non-anchored', fontsize=9)
ax.set_ylabel('gain, anchored', fontsize=9)
below = (S.gain_a < S.gain_n).mean() * 100
ax.set_title(f'{len(S)} speed cells — {below:.0f}% below identity', fontsize=9.5, loc='left')

ax = axes[1, 0]
d = D.dropna(subset=['dec_r_a', 'dec_r_n'])
for i in range(len(d)):
    ax.plot([0, 1], [d.dec_r_n.iloc[i], d.dec_r_a.iloc[i]], color='0.75', lw=.6, zorder=1)
ax.scatter(np.zeros(len(d)), d.dec_r_n, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
ax.scatter(np.ones(len(d)), d.dec_r_a, s=14, color=ANCH_COLOR, lw=0, zorder=2)
for x, v, c in ((0, d.dec_r_n, NONANCH_COLOR), (1, d.dec_r_a, ANCH_COLOR)):
    ax.plot([x - .18, x + .18], [v.median()] * 2, color=c, lw=2.4, zorder=3)
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
ax.set_xlim(-.4, 1.4)
ax.set_ylabel('speed decoding r (pred, true)', fontsize=9)
ax.set_title(f'{len(d)} sessions — p = {wilcoxon(d.dec_r_a, d.dec_r_n).pvalue:.2g}',
             fontsize=9.5, loc='left')

# E, F: the residual-speed analysis -- position partialled out of both sides
V = pd.read_csv(f'{OUT}/residual_speed.csv')
W = V.pivot_table(index=['mouse', 'day', 'cluster'], columns='state',
                  values=['slope_cms', 'sres_sd']).dropna()


def pairpanel(ax, va, vn, ylab, title):
    for i in range(len(va)):
        ax.plot([0, 1], [vn[i], va[i]], color='0.75', lw=.6, zorder=1)
    ax.scatter(np.zeros(len(va)), vn, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
    ax.scatter(np.ones(len(va)), va, s=14, color=ANCH_COLOR, lw=0, zorder=2)
    for x, v, c in ((0, vn, NONANCH_COLOR), (1, va, ANCH_COLOR)):
        ax.plot([x - .18, x + .18], [np.median(v)] * 2, color=c, lw=2.4, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
    ax.set_xlim(-.4, 1.4)
    ax.set_ylabel(ylab, fontsize=9)
    ax.set_title(f'{title} — p = {wilcoxon(va, vn).pvalue:.2g}', fontsize=9.5, loc='left')


sd = W['sres_sd'].groupby(level=['mouse', 'day']).median().dropna()
pairpanel(axes[1, 1], sd['a'].values, sd['n'].values,
          'SD of unexpected speed (cm/s)', f'{len(sd)} sessions')
sl = W['slope_cms'].groupby(level=['mouse', 'day']).median().dropna()
pairpanel(axes[1, 2], sl['a'].values, sl['n'].values,
          'Hz per cm/s unexpected speed', f'{len(sl)} sessions')

for a in axes.ravel():
    a.spines[['top', 'right']].set_visible(False)
    a.tick_params(labelsize=7.5)
plt.tight_layout()
out = f'{FIG}/fig6_speed_tuning_by_state.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)


## Spike–theta phase locking: the entrainment prediction fails

Theta amplitude decouples from running speed when anchored while rate coding of
speed does not change. That was read as anchoring changing how strongly the
population is **entrained** to theta rather than what cells encode — and that
word makes a prediction: phase locking should weaken when anchored, and its
speed dependence should weaken with it.

**Neither happens.** Across 7,542 cells in 59 sessions, with spike counts and
speed distributions matched between states:

| | non-anchored | anchored | p |
|---|---|---|---|
| MRL | 0.1425 | 0.1414 | **0.79** |
| speed dependence (fast − slow tercile) | +0.0286 | +0.0386 | 1.3 × 10⁻⁵ |
| preferred phase shift | — | +1.1° | — |

Locking strength is unchanged (30/59 sessions lower when anchored — chance), the
preferred phase barely moves, and the speed dependence of locking goes the
*opposite* way to the prediction: locking discriminates fast from slow running
**more** strongly when anchored.

The significant speed-dependence result doubles as a positive control: the
pipeline detects a real state difference, so the null on MRL is not a dead
measurement.

**What this leaves.** Three things are now known to be unchanged between states
— rate coding of speed, sensitivity to unexpected speed, and spike–theta timing.
What changes is the amplitude of the field potential and its scaling with speed.
Since LFP theta amplitude reflects summed synaptic and dendritic currents rather
than local spiking, the locus is most likely the magnitude of the speed-scaled
synaptic drive arriving in MEC, upstream of the local circuit — whose spike
timing relative to theta is preserved. "Entrainment" is the wrong word and has
been removed from the paper.

Two confounds are handled by one matching step, and both would otherwise run in
the dangerous direction: MRL is biased upward at small n and anchored firing
rates are ~7% lower (so unmatched counts would make anchored cells look *more*
locked), and locking depends on speed, which differs between states. Each
cluster takes the phase of its own extremum channel's group — theta phase
rotates through the entorhinal layers, so a session average would destroy the
quantity being measured.

Run `phase_locking.py` (~28 min) then `phase_locking_fig.py`.

In [ ]:
"""Spike-theta phase locking by anchoring state.

The LFP decouples from running speed when the population is anchored while the
cells' rate coding of speed does not change, which says the state alters how
strongly population activity is ENTRAINED to theta rather than what cells
encode. That word makes a prediction this tests directly: spike-theta phase
locking should weaken when anchored, and its speed dependence should weaken
with it. If locking is unchanged, "entrainment" is the wrong description and the
effect lives in synaptic currents rather than spike timing.

Each cluster is assigned the theta phase of its OWN extremum channel's group
(`extremum_channel` gives 'CH244', group names list their eight contacts), not a
session average: theta phase rotates through the entorhinal layers, so averaging
phase across groups would destroy exactly the quantity being measured.

TWO CONFOUNDS, both handled by one matching step.

  Spike count. The mean resultant length is biased upward at small n, and
  anchored firing rates are ~7% lower, which alone would make anchored cells
  look MORE locked. Counts are equalised between states.

  Speed. Locking depends on running speed, and the states differ in their speed
  distribution. Spikes are matched across speed deciles before counts are
  equalised.

MRL is then averaged over NDRAW random subsamples so the estimate does not ride
on one draw. The speed dependence is measured as MRL in the top speed tercile
minus the bottom, computed within the matched set with counts equalised in each
tercile, so it carries the same protections.
"""
import os, sys, json, time, warnings
import h5py, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import pynapple as nap
from scipy.stats import wilcoxon
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
                   'AnchorDynamics2026')
from lfp_batch import LFP_ROOT, FS, vr_paths, clip_trials

_g = {}
for _c in json.load(open('/Users/harryclark/Documents/spatial-manifolds/scripts/'
                         'figures/AnchorDynamics2026/lick_raster_by_trial_type.ipynb'))['cells']:
    if _c['cell_type'] != 'code':
        continue
    _s = ''.join(_c['source'])
    if _s.startswith('MOUSE, DAY') or 'INV = pd.DataFrame' in _s:
        continue
    exec(compile(_s, '<nb>', 'exec'), _g)
globals().update({k: v for k, v in _g.items() if not k.startswith('__')})

OUT = '/Users/harryclark/Documents/spatial-manifolds/data/lfp'
RUN_SPEED = 3.0
MINSPK = 300          # per state, after matching
NDRAW = 20


def mrl(ph):
    return float(np.abs(np.mean(np.exp(1j * ph))))


def matched_mrl(pa, pn, rng, ndraw=NDRAW):
    """MRL for each state at equal spike count, averaged over draws."""
    n = min(len(pa), len(pn))
    if n < MINSPK:
        return np.nan, np.nan, 0
    va = np.mean([mrl(rng.choice(pa, n, replace=False)) for _ in range(ndraw)])
    vn = np.mean([mrl(rng.choice(pn, n, replace=False)) for _ in range(ndraw)])
    return va, vn, n


def speed_match(sa, sn, rng, nq=10):
    """Indices into each state's spikes with matched speed distributions."""
    q = np.quantile(np.r_[sa, sn], np.linspace(0, 1, nq + 1))
    q[0] -= 1; q[-1] += 1
    ka, kn = [], []
    for lo, hi in zip(q[:-1], q[1:]):
        ia = np.where((sa >= lo) & (sa < hi))[0]
        iN = np.where((sn >= lo) & (sn < hi))[0]
        k = min(len(ia), len(iN))
        if k:
            ka.append(rng.choice(ia, k, replace=False))
            kn.append(rng.choice(iN, k, replace=False))
    if not ka:
        return np.array([], int), np.array([], int)
    return np.concatenate(ka), np.concatenate(kn)


def run_session(mo, dy, state, rng):
    lf = f'{LFP_ROOT}/M{mo}/D{dy}/VR/sub-M{mo}_ses-D{dy}_typ-VR_beh.nwb'
    if not os.path.exists(lf):
        return None
    f = h5py.File(lf, 'r')
    if 'processing/ecephys/Processed/theta/data' not in f:
        f.close(); return None
    names = [n.decode() for n in f['general/extracellular_ephys/electrodes/channel_name'][:]]
    ch2g = {c: i for i, n in enumerate(names) for c in n.split('_')}

    bp, cp = vr_paths(mo, dy)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
    ids = spatial_cell_ids(mo, dy, clusters, classes=('GC', 'NGS', 'NS'))
    if len(ids) < 5:
        f.close(); return None
    ec = clusters.metadata['extremum_channel']
    grp = {c: ch2g.get(str(ec[c]), -1) for c in ids}
    use = [c for c in ids if grp[c] >= 0]
    if not use:
        f.close(); return None

    cols = sorted(set(grp[c] for c in use))
    back = {g: i for i, g in enumerate(cols)}
    PH = f['processing/ecephys/Processed/theta/data'][:, cols].astype(np.float32)
    f.close()

    n = PH.shape[0]
    t = np.arange(n) / FS
    S = beh['S']
    spd = np.asarray(S.values)[np.searchsorted(np.asarray(S.index), t).clip(0, len(S) - 1)]
    starts = trials.start.values.astype(float)
    ends = trials.end.values.astype(float)
    nums = trials.number.values.astype(int)
    k = np.searchsorted(starts, t) - 1
    inside = (k >= 0) & (k < len(starts))
    k = k.clip(0, len(starts) - 1)
    inside &= t <= ends[k]
    lab = np.where(inside, [state.get(int(u), np.nan) for u in nums[k]], np.nan)
    ok = (spd >= RUN_SPEED) & np.isfinite(lab)

    rows = []
    for c in use:
        sp = np.asarray(clusters[c].index, dtype=float)
        i = np.round(sp * FS).astype(int)
        i = i[(i >= 0) & (i < n)]
        i = i[ok[i]]
        if len(i) < 2 * MINSPK:
            continue
        ph = PH[i, back[grp[c]]].astype(float)
        sv, st = spd[i], lab[i] > .5
        pa, pn = ph[st], ph[~st]
        sa, sn = sv[st], sv[~st]
        ia, iN = speed_match(sa, sn, rng)
        if len(ia) < MINSPK or len(iN) < MINSPK:
            continue
        pa, pn, sa, sn = pa[ia], pn[iN], sa[ia], sn[iN]
        va, vn, nm = matched_mrl(pa, pn, rng)
        d = dict(mouse=mo, day=dy, cluster=int(c), layer='', n_matched=nm,
                 mrl_a=va, mrl_n=vn,
                 phase_a=float(np.angle(np.mean(np.exp(1j * pa)))),
                 phase_n=float(np.angle(np.mean(np.exp(1j * pn)))),
                 rate_a=float(len(ph[st])), rate_n=float(len(ph[~st])))
        # speed dependence: top minus bottom tercile, counts equalised inside each
        q = np.quantile(np.r_[sa, sn], [1 / 3, 2 / 3])
        for tag, lo, hi in (('lo', -np.inf, q[0]), ('hi', q[1], np.inf)):
            ma, mn = (sa >= lo) & (sa < hi), (sn >= lo) & (sn < hi)
            if ma.sum() >= MINSPK and mn.sum() >= MINSPK:
                xa, xn, _ = matched_mrl(pa[ma], pn[mn], rng, ndraw=10)
                d[f'mrl_{tag}_a'], d[f'mrl_{tag}_n'] = xa, xn
        rows.append(d)
    return pd.DataFrame(rows)


def one_cell(mo, dy, state, cl, rng):
    """The matched spike phases of a single cluster, for plotting."""
    lf = f'{LFP_ROOT}/M{mo}/D{dy}/VR/sub-M{mo}_ses-D{dy}_typ-VR_beh.nwb'
    f = h5py.File(lf, 'r')
    names = [n.decode() for n in f['general/extracellular_ephys/electrodes/channel_name'][:]]
    ch2g = {c: i for i, n in enumerate(names) for c in n.split('_')}
    bp, cp = vr_paths(mo, dy)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    trials, orig = clip_trials(beh['trials'].as_dataframe(), clusters)
    g = ch2g[str(clusters.metadata['extremum_channel'][cl])]
    PH = f['processing/ecephys/Processed/theta/data'][:, g].astype(np.float32)
    f.close()
    n = len(PH)
    t = np.arange(n) / FS
    S = beh['S']
    spd = np.asarray(S.values)[np.searchsorted(np.asarray(S.index), t).clip(0, len(S) - 1)]
    starts, ends = trials.start.values.astype(float), trials.end.values.astype(float)
    nums = trials.number.values.astype(int)
    k = np.searchsorted(starts, t) - 1
    inside = (k >= 0) & (k < len(starts))
    k = k.clip(0, len(starts) - 1)
    inside &= t <= ends[k]
    lab = np.where(inside, [state.get(int(u), np.nan) for u in nums[k]], np.nan)
    ok = (spd >= RUN_SPEED) & np.isfinite(lab)
    i = np.round(np.asarray(clusters[cl].index, dtype=float) * FS).astype(int)
    i = i[(i >= 0) & (i < n)]
    i = i[ok[i]]
    ph, sv, stt = PH[i].astype(float), spd[i], lab[i] > .5
    ia, iN = speed_match(sv[stt], sv[~stt], rng)
    pa, pn = ph[stt][ia], ph[~stt][iN]
    m = min(len(pa), len(pn))                  # equal counts, as in the batch
    return rng.choice(pa, m, replace=False), rng.choice(pn, m, replace=False)


if __name__ == '__main__':
    T = pd.read_csv(f'{OUT}/theta_frequency.csv')
    extra = f'{OUT}/theta_frequency_extra.csv'
    if os.path.exists(extra):
        T = pd.concat([T, pd.read_csv(extra)], ignore_index=True)
    sess = sorted(set(zip(T.mouse, T.day)))
    out, t0 = [], time.time()
    for i, (mo, dy) in enumerate(sess, 1):
        st = T[(T.mouse == mo) & (T.day == dy)]
        rng = np.random.default_rng(abs(hash((mo, dy))) % 2**32)
        try:
            R = run_session(int(mo), int(dy),
                            dict(zip(st.trial.astype(int), st.anch.astype(bool))), rng)
        except Exception as e:
            print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True); continue
        if R is None or not len(R):
            continue
        out.append(R)
        print(f'[{i}/{len(sess)}] M{mo}D{dy}: {len(R)} cells, {time.time()-t0:.0f}s',
              flush=True)
    A = pd.concat(out, ignore_index=True)
    A.to_csv(f'{OUT}/phase_locking.csv', index=False)

    print(f'\n{len(A)} cells, {A.groupby(["mouse","day"]).ngroups} sessions, '
          f'{A.n_matched.median():.0f} matched spikes per state (median)')
    g = A.groupby(['mouse', 'day'])[['mrl_a', 'mrl_n']].median().dropna()
    print(f'  MRL (count- and speed-matched)  anchored {g.mrl_a.median():.4f}  '
          f'non {g.mrl_n.median():.4f}  '
          f'({100*(g.mrl_a.median()-g.mrl_n.median())/g.mrl_n.median():+.1f}%)  '
          f'p = {wilcoxon(g.mrl_a, g.mrl_n).pvalue:.3g}  '
          f'({(g.mrl_a < g.mrl_n).sum()}/{len(g)} lower when anchored)')
    D = A.dropna(subset=['mrl_lo_a', 'mrl_hi_a', 'mrl_lo_n', 'mrl_hi_n']).copy()
    D['dep_a'] = D.mrl_hi_a - D.mrl_lo_a
    D['dep_n'] = D.mrl_hi_n - D.mrl_lo_n
    gd = D.groupby(['mouse', 'day'])[['dep_a', 'dep_n']].median().dropna()
    print(f'  speed dependence (hi - lo MRL)  anchored {gd.dep_a.median():+.4f}  '
          f'non {gd.dep_n.median():+.4f}  p = {wilcoxon(gd.dep_a, gd.dep_n).pvalue:.3g}  '
          f'({len(gd)} sessions, {len(D)} cells)')
    ph = A.dropna(subset=['phase_a', 'phase_n'])
    sh = np.angle(np.mean(np.exp(1j * (ph.phase_a - ph.phase_n))))
    print(f'  preferred phase shift (anchored - non): {np.degrees(sh):+.1f} deg')


In [ ]:
"""Figure: spike-theta phase locking by anchoring state.

A  phase histogram for one cell, both states, spike counts equalised
B  session medians of MRL, paired (count- and speed-matched)
C  every cell, anchored MRL against non-anchored
D  speed dependence of locking (top minus bottom speed tercile), paired

The direct test of the entrainment reading: rate coding of speed is unchanged
between states, so if locking also turns out unchanged then "entrainment" is
the wrong word for what anchoring does to the LFP.
"""
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
from spatial_manifolds.anchoring import ANCH_COLOR, NONANCH_COLOR

s = open('/private/tmp/claude-501/-Users-harryclark-Documents-spatial-manifolds/'
         '35f3d8d1-3ead-4d48-8089-491c15b000b7/scratchpad/phase_locking.py').read()
exec(s[:s.index(chr(10) + 'if __name__')])

plt.rcParams['font.family'] = 'Arial'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
A = pd.read_csv(f'{OUT}/phase_locking.csv').dropna(subset=['mrl_a', 'mrl_n'])

# example: a well-locked cell at the median change, selection not conditioned on
# the direction of that change
cand = A[(A.n_matched > 3000) & (A[['mrl_a', 'mrl_n']].max(1) > .08)].copy()
cand['d'] = cand.mrl_a - cand.mrl_n
ex = cand.iloc[(cand.d - (A.mrl_a - A.mrl_n).median()).abs().argsort()].iloc[0]
MO, DY, CL = int(ex.mouse), int(ex.day), int(ex.cluster)
print(f'example M{MO}D{DY} cluster {CL}: MRL {ex.mrl_a:.3f} anchored vs {ex.mrl_n:.3f}')

T = pd.read_csv(f'{OUT}/theta_frequency.csv')
extra = f'{OUT}/theta_frequency_extra.csv'
if os.path.exists(extra):
    T = pd.concat([T, pd.read_csv(extra)], ignore_index=True)
st = T[(T.mouse == MO) & (T.day == DY)]
pa, pn = one_cell(MO, DY, dict(zip(st.trial.astype(int), st.anch.astype(bool))), CL,
                  np.random.default_rng(0))

fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.0))

ax = axes[0, 0]
# phase in this dataset runs 0..2pi, not -pi..pi; two cycles are drawn because a
# single cycle cut at an arbitrary point hides whether the peak is real
bins = np.linspace(0, 2 * np.pi, 25)
ctr = bins[:-1] + np.diff(bins) / 2
for v, c, lb in ((pa, ANCH_COLOR, 'anchored'), (pn, NONANCH_COLOR, 'non-anchored')):
    h, _ = np.histogram(np.mod(v, 2 * np.pi), bins=bins)
    h = h / h.sum()
    ax.plot(np.r_[ctr, ctr + 2 * np.pi], np.r_[h, h], color=c, lw=2, label=lb)
ax.legend(fontsize=7.5, frameon=False)
ax.set_title(f'M{MO} D{DY} cluster {CL} — MRL {ex.mrl_a:.3f} vs {ex.mrl_n:.3f}',
             fontsize=9.5, loc='left')
ax.set_xlabel('Theta phase (two cycles)', fontsize=9)
ax.set_ylabel('Spike density', fontsize=9)
ax.set_xlim(0, 4 * np.pi)
ax.set_xticks([0, np.pi, 2 * np.pi, 3 * np.pi, 4 * np.pi])
ax.set_xticklabels(['0', 'π', '2π', '3π', '4π'])

ax = axes[0, 1]
g = A.groupby(['mouse', 'day'])[['mrl_a', 'mrl_n']].median().dropna()
for i in range(len(g)):
    ax.plot([0, 1], [g.mrl_n.iloc[i], g.mrl_a.iloc[i]], color='0.75', lw=.6, zorder=1)
ax.scatter(np.zeros(len(g)), g.mrl_n, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
ax.scatter(np.ones(len(g)), g.mrl_a, s=14, color=ANCH_COLOR, lw=0, zorder=2)
for x, v, c in ((0, g.mrl_n, NONANCH_COLOR), (1, g.mrl_a, ANCH_COLOR)):
    ax.plot([x - .18, x + .18], [v.median()] * 2, color=c, lw=2.4, zorder=3)
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
ax.set_xlim(-.4, 1.4)
ax.set_ylabel('MRL (count- and speed-matched)', fontsize=9)
ax.set_title(f'{len(g)} sessions — p = {wilcoxon(g.mrl_a, g.mrl_n).pvalue:.2g}',
             fontsize=9.5, loc='left')

ax = axes[1, 0]
lim = np.nanpercentile(np.r_[A.mrl_a, A.mrl_n], [0, 99])
ax.scatter(A.mrl_n, A.mrl_a, s=6, color='0.45', alpha=.35, lw=0)
ax.plot(lim, lim, color='0.2', lw=.9, ls='--')
ax.set_xlim(*lim); ax.set_ylim(*lim)
ax.set_xlabel('MRL, non-anchored', fontsize=9)
ax.set_ylabel('MRL, anchored', fontsize=9)
ax.set_title(f'{len(A)} cells — {(A.mrl_a < A.mrl_n).mean()*100:.0f}% below identity',
             fontsize=9.5, loc='left')

ax = axes[1, 1]
D = A.dropna(subset=['mrl_lo_a', 'mrl_hi_a', 'mrl_lo_n', 'mrl_hi_n']).copy()
D['dep_a'], D['dep_n'] = D.mrl_hi_a - D.mrl_lo_a, D.mrl_hi_n - D.mrl_lo_n
gd = D.groupby(['mouse', 'day'])[['dep_a', 'dep_n']].median().dropna()
for i in range(len(gd)):
    ax.plot([0, 1], [gd.dep_n.iloc[i], gd.dep_a.iloc[i]], color='0.75', lw=.6, zorder=1)
ax.scatter(np.zeros(len(gd)), gd.dep_n, s=14, color=NONANCH_COLOR, lw=0, zorder=2)
ax.scatter(np.ones(len(gd)), gd.dep_a, s=14, color=ANCH_COLOR, lw=0, zorder=2)
for x, v, c in ((0, gd.dep_n, NONANCH_COLOR), (1, gd.dep_a, ANCH_COLOR)):
    ax.plot([x - .18, x + .18], [v.median()] * 2, color=c, lw=2.4, zorder=3)
ax.axhline(0, color='0.5', lw=.7, ls=':')
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch', 'anchored'], fontsize=8)
ax.set_xlim(-.4, 1.4)
ax.set_ylabel('MRL, fast minus slow tercile', fontsize=9)
ax.set_title(f'{len(gd)} sessions — p = {wilcoxon(gd.dep_a, gd.dep_n).pvalue:.2g}',
             fontsize=9.5, loc='left')

for a in axes.ravel():
    a.spines[['top', 'right']].set_visible(False)
    a.tick_params(labelsize=7.5)
plt.tight_layout()
out = f'{FIG}/fig6_phase_locking_by_state.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight')
print('saved', out)
